# Dashboard Royale: Gilded Ace floor analysis (Team 116)

**The situation.** The general manager plans to replace the 10 machines that earned the least between January 1 and June 30, 2025.
**Our question.** Is low earnings really the machine's fault, or is it **luck, location, or a leak** (downtime or payout)?

**Contents**
1. Setup and floor overview
2. Q1: Which machines are truly underperforming?
3. Q2: What changed in demand, and why?
4. Q3: How much play was lost to downtime?
5. Q4: What did tracked players wager, get back and net?
6. Q5: What did the comp program cost, and did anything change?
7. Summary and direction for Q6 (the $100,000 plan)

The forecast model is in a separate notebook, `model.ipynb`.

*Conventions:* only active machine-days are used (1,018 off-floor rows excluded); ratios are always **ratio of sums**; money columns stored as exact decimals are converted to floats by `load()`.

## 1. Setup and floor overview

In [161]:
import pandas as pd
import decimal

def load(path):
    df = pd.read_parquet(path)
    for c in df.columns:
        if df[c].dtype == object and df[c].dropna().map(type).eq(decimal.Decimal).any():
            df[c] = df[c].astype(float)
    return df

prop  = load("data/1_prepared_views/property_day.parquet")
daily = load("data/1_prepared_views/prepared_daily_view.parquet")

**Charts.** Every chart is Plotly. `save()` shows it here and writes `<name>.html` next to `index.html`, which the dashboard loads in an `<iframe>`. Run the whole notebook top to bottom to refresh every chart file.

In [162]:
# Plotly setup: one dark "Gilded Ace" style for every chart. save() shows each chart here AND writes it
# to its own HTML file next to index.html, so the dashboard can show it in an <iframe>.
import datetime
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

BG, WHITE = "#11244D", "#F2F2F2"
CYAN, GOLD, RED, GREEN, PURPLE, ORANGE = "#00D8FF", "#EAA244", "#E4572E", "#3BB273", "#B07CFF", "#FF9F1C"
GREY, FAINT, GRID = "#8A94A6", "rgba(255,255,255,0.22)", "rgba(255,255,255,0.08)"
TIER_COLORS = {"Base": GREY, "Gold": GOLD, "Platinum": PURPLE}

pio.templates["gilded"] = go.layout.Template(layout=dict(
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family="Inter, sans-serif", color=WHITE, size=13),
    colorway=[CYAN, GOLD, RED, GREEN, PURPLE, ORANGE, "#FF8FB1", "#7FD1B9", GREY],
    xaxis=dict(gridcolor=GRID, zerolinecolor=FAINT, linecolor=FAINT, automargin=True),
    yaxis=dict(gridcolor=GRID, zerolinecolor=FAINT, linecolor=FAINT, automargin=True),
    legend=dict(bgcolor="rgba(0,0,0,0)"),
    margin=dict(l=60, r=30, t=70, b=60),
))
pio.templates.default = "plotly_dark+gilded"

CHART_DIR = "."   # the folder that holds index.html
PAGE_STYLE = f"<style>html,body{{margin:0;height:100%;background:{BG};}}</style>"

def save(fig, name, height=450, slider=None):
    """Show the chart in the notebook, then write CHART_DIR/<name>.html sized to fill its iframe.
    slider: zoom slider under the x-axis. None = automatic: every single-panel chart with a date axis gets one."""
    fig.update_layout(height=height)
    if slider is None:
        single_x = not any(k.startswith("xaxis") and k != "xaxis" for k in fig.layout.to_plotly_json())
        first_x = next((t.x for t in fig.data if getattr(t, "x", None) is not None and len(t.x)), None)
        first_val = next((v for v in first_x if v is not None), None) if first_x is not None else None
        if isinstance(first_val, str):                       # dates stored as text, e.g. "2025-01-01"
            try:
                first_val = pd.Timestamp(first_val) if len(first_val) >= 8 and first_val[:4].isdigit() else None
            except ValueError:
                first_val = None
        slider = single_x and isinstance(first_val, (datetime.date, np.datetime64, pd.Timestamp))
    print(f"zoom slider: {'yes' if slider else 'no'}")
    # Bars with labels outside them: leave headroom past the longest bar so labels like "+23%" never get cut off
    for orient, axis in [("h", "xaxis"), ("v", "yaxis")]:
        bars = [t for t in fig.data if t.type == "bar" and t.textposition == "outside"
                and (t.orientation or "v") == orient]
        if bars and fig.layout[axis].range is None:
            vals = np.concatenate([np.asarray(t.x if orient == "h" else t.y, dtype=float) for t in bars])
            lo, hi = np.nanmin(vals), np.nanmax(vals)
            fig.update_layout({axis: dict(range=[min(0, lo) * 1.18, max(0, hi) * 1.18])})
            for t in bars:
                t.cliponaxis = False
    if slider:
        # A thick, gold-framed zoom strip so it's obvious it can be dragged
        fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.13, bgcolor="rgba(234,162,68,0.10)",
                                          bordercolor=GOLD, borderwidth=2))
        legend_at_bottom = (fig.layout.legend.orientation == "h" and
                            (fig.layout.legend.y if fig.layout.legend.y is not None else 1) < 0)
        if legend_at_bottom:                          # a bottom legend would collide with the slider: move it on top
            fig.update_layout(legend=dict(x=0, y=1.02, yanchor="bottom"))
            if fig.layout.title.text:                 # and lift the title clear of it
                fig.update_layout(title=dict(y=0.98, yref="container", yanchor="top"), margin_t=110)
        fig.update_layout(margin_b=60)
    fig.show()
    page = go.Figure(fig)
    page.update_layout(height=None, width=None, autosize=True)
    # Dashboard copy: no inner title (the card above the chart already has one, and long titles get cut off)
    legend_on_top = page.layout.legend.y is not None and page.layout.legend.y >= 1
    page.update_layout(title=dict(text=""), margin_t=60 if legend_on_top else 40)
    html = page.to_html(include_plotlyjs="cdn", full_html=True, default_width="100%", default_height="100%",
                        config={"displayModeBar": False, "responsive": True})
    with open(f"{CHART_DIR}/{name}.html", "w", encoding="utf-8") as f:
        f.write(html.replace("<head>", "<head>" + PAGE_STYLE, 1))
    print(f"saved {name}.html")

def date_line(fig, date, label="", color=RED):
    """Dashed vertical line on a date axis (add_vline's built-in label breaks on dates)."""
    fig.add_shape(type="line", x0=date, x1=date, y0=0, y1=1, yref="paper", line=dict(color=color, dash="dash"))
    if label:
        fig.add_annotation(x=date, y=1, yref="paper", text=label, showarrow=False, yanchor="bottom",
                           font=dict(color=color, size=11))

def grey_lines(df, exclude=()):
    """All columns of df as ONE faint line trace (None breaks between machines), so hundreds of lines stay fast."""
    xs, ys = [], []
    for c in df.columns.drop(list(exclude), errors="ignore"):
        xs += list(df.index) + [None]
        ys += list(df[c]) + [None]
    return xs, ys

In [163]:
daily = load("data/1_prepared_views/prepared_daily_view.parquet")
prop  = load("data/1_prepared_views/property_day.parquet")
d = daily[daily["active"]]

fig = go.Figure()
fig.add_scatter(x=prop["gaming_date"], y=prop["coin_in_usd"], name="Coin-in (wagered)", line=dict(color=CYAN, width=2))
fig.add_scatter(x=prop["gaming_date"], y=prop["theo_usd"] * 10, name="Theo × 10 (expected win)", line=dict(color=GOLD, width=2))

# Label each event spike at its busiest day. Super Bowl and the Winter Tournament are two weeks apart,
# so their labels go at different heights and point in opposite directions to avoid overlapping.
peak = prop.set_index(pd.to_datetime(prop["gaming_date"]))["coin_in_usd"].sort_index()
for start, end, label, ay, anchor in [("2025-02-07", "2025-02-09", "Super Bowl",         -62, "right"),
                                      ("2025-02-21", "2025-02-23", "Winter Tournament",  -30, "left"),
                                      ("2025-03-20", "2025-03-22", "College basketball", -30, "center"),
                                      ("2025-05-16", "2025-05-18", "EDC weekend",        -30, "center")]:
    window = peak.loc[start:end]
    fig.add_annotation(x=window.idxmax(), y=window.max(), text=label, showarrow=True, arrowhead=0, ax=0, ay=ay,
                       xanchor=anchor, font=dict(size=11, color=GOLD), arrowcolor=GOLD)
# No title inside the chart (the dashboard card already has one); legend sits neatly above the plot
fig.update_layout(yaxis_tickprefix="$", hovermode="x unified", margin=dict(t=60),
                  legend=dict(orientation="h", x=0, y=1.02, yanchor="bottom"))
save(fig, "whole_floor_performance", height=460)


zoom slider: yes


saved whole_floor_performance.html


- **Spike 1:** Super Bowl weekend (Feb 7–9)
- **Spike 2:** Gilded Ace Winter Tournament (Feb 21–23)
- **Spike 3:** College basketball tournament opening rounds (Mar 20–22)
- **Spike 4:** EDC Las Vegas festival weekend (May 16–18)

**What the overview shows.** Play follows a strong weekly rhythm (weekends about 70% busier than weekdays) with no long-term decline. The four big spikes match events, which are analysed under Q2. Theo (expected win) tracks coin-in at about 9.5%, which is the floor's average designed hold.

---
## 2. Q1: Which machines are truly underperforming?

We test each explanation in turn:
- **Exposure:** was the machine on the floor all six months?
- **Availability:** was it broken?
- **Luck:** is actual win far from theo, beyond normal randomness?
- **Location vs game vs machine:** regression with zone, bank, game and day effects
- **Natural experiments:** machines that were replaced or moved

### 2.1 Actual vs expected win per machine

In [164]:
m = d.groupby("machine_id").agg(house_win_usd=("house_win_usd", "sum"), theo_usd=("theo_usd", "sum"),
                                available_hours=("available_hours", "sum"),
                                theme=("theme", "first"), zone=("zone_name", "last"))

fig = go.Figure(go.Histogram(x=m["house_win_usd"], nbinsx=40, marker_color=CYAN))
fig.update_layout(title="House win per machine (6 months)", xaxis_title="USD", yaxis_title="Machines",
                  xaxis_tickprefix="$", bargap=0.05)
save(fig, "house_win_histogram", height=380)

lim = [m["theo_usd"].min(), m["theo_usd"].max()]
fig = go.Figure()
fig.add_scatter(x=m["theo_usd"], y=m["house_win_usd"], mode="markers", name="Machine",
                marker=dict(size=6, color=CYAN, opacity=0.75),
                customdata=np.stack([m.index, m["theme"], m["zone"]], axis=1),
                hovertemplate="%{customdata[0]} · %{customdata[1]} · %{customdata[2]}<br>"
                              "Theo $%{x:,.0f}<br>Actual $%{y:,.0f}<extra></extra>")
fig.add_scatter(x=lim, y=lim, mode="lines", name="actual = expected", line=dict(color=RED, dash="dash"))
fig.update_layout(title="Actual vs expected win, each dot = a machine", xaxis_title="Theo (expected win)",
                  yaxis_title="Actual house win", xaxis_tickprefix="$", yaxis_tickprefix="$",
                  legend=dict(orientation="h", y=1.08))
save(fig, "theo_vs_actual", height=430)

zoom slider: no


saved house_win_histogram.html
zoom slider: no


saved theo_vs_actual.html


### 2.2 The two humps: Back Forty and four low-earning games

In [165]:
daily = load("data/1_prepared_views/prepared_daily_view.parquet")
d = daily[daily["active"]]

m = d.groupby("machine_id").agg(
    house_win=("house_win_usd", "sum"),
    zone=("zone_name", "last"),
    theme=("theme", "first"),
)

low_games = ["Aurora Link", "Comet Cash", "Lucky Lanterns", "Royal Draw"]
m["category"] = "Other zones, other games"
m.loc[m["theme"].isin(low_games), "category"] = "Low-only games"
m.loc[m["zone"] == "Back Forty", "category"] = "Back Forty zone"

BINS = dict(start=0, end=190_000, size=5_000)

def humps(traces, title, name):
    fig = go.Figure(traces)
    fig.add_vline(x=77_000, line_dash="dash", line_color=RED, annotation_text="$77K split", annotation_position="top")
    fig.update_layout(barmode="stack", bargap=0.05, title=title, xaxis_title="House win per machine, 6 months (USD)",
                      yaxis_title="Number of machines", xaxis_tickprefix="$", legend=dict(orientation="h", y=-0.22))
    save(fig, name, height=430)

# 1) Original histogram with the split line
humps([go.Histogram(x=m["house_win"], xbins=BINS, marker_color=CYAN, name="All machines")],
      "House win per machine: a low hump and a high hump", "house_win_split")

# 2) Colored by Back Forty / low-only games / everything else
groups = {"Back Forty zone": RED, "Low-only games": GOLD, "Other zones, other games": GREY}
humps([go.Histogram(x=m.loc[m["category"] == g, "house_win"], xbins=BINS, name=g, marker_color=c)
       for g, c in groups.items()],
      "The two humps: the low hump is Back Forty and four low-earning games", "two_humps")

# 3) Colored by every zone, ordered from weakest to strongest median
zone_order = m.groupby("zone")["house_win"].median().sort_values().index
other_colors = [CYAN, GREEN, PURPLE, ORANGE, "#FF8FB1", "#7FD1B9", GOLD]
zone_colors, i = {}, 0
for z in zone_order:
    if z == "Back Forty":
        zone_colors[z] = RED
    else:
        zone_colors[z] = other_colors[i % len(other_colors)]
        i += 1
humps([go.Histogram(x=m.loc[m["zone"] == z, "house_win"], xbins=BINS, name=z, marker_color=zone_colors[z])
       for z in zone_order],
      "Colored by zone (weakest median first)", "house_win_by_zone")

print(m["category"].value_counts())

zoom slider: no


saved house_win_split.html
zoom slider: no


saved two_humps.html
zoom slider: no


saved house_win_by_zone.html
category
Other zones, other games    193
Low-only games               69
Back Forty zone              48
Name: count, dtype: int64


### 2.3 Demand by zone × game (coin-in per working hour)

In [166]:
daily = load("data/1_prepared_views/prepared_daily_view.parquet")
gm    = load("data/2_machine_decision/game_math.parquet")
d = daily[daily["active"]]

# 1) Game settings: are the low-only games just smaller-bet games?
print(gm[["theme", "configured_rtp", "volatility_class", "min_bet_cents", "max_bet_cents"]]
        .sort_values("max_bet_cents").to_string(index=False), "\n")

# 2) Fair demand measure: coin-in per available hour, for each zone x game
zg = d.groupby(["zone_name", "theme"]).agg(coin=("coin_in_usd", "sum"),
                                           hrs=("available_hours", "sum"),
                                           machines=("machine_id", "nunique"))
zg["rate"] = zg["coin"] / zg["hrs"]
rate = zg["rate"].unstack()
count = zg["machines"].unstack()

# Order zones and games from weakest to strongest
zone_rank = d.groupby("zone_name").apply(lambda g: g["coin_in_usd"].sum() / g["available_hours"].sum()).sort_values()
game_rank = d.groupby("theme").apply(lambda g: g["coin_in_usd"].sum() / g["available_hours"].sum()).sort_values()
rate = rate.loc[zone_rank.index, game_rank.index]
count = count.loc[zone_rank.index, game_rank.index]

# 3) Heatmap: each cell = $ wagered per available hour; number = machines in that cell
text = [["" if pd.isna(rate.iat[i, j]) else f"${rate.iat[i, j]:,.0f}<br>({int(count.iat[i, j])})"
         for j in range(rate.shape[1])] for i in range(rate.shape[0])]
fig = go.Figure(go.Heatmap(z=rate.values, x=list(rate.columns), y=list(rate.index), text=text,
                           texttemplate="%{text}", textfont=dict(size=10, color="black"),
                           colorscale="RdYlGn", hoverongaps=False, colorbar=dict(title="$ / hr"),
                           hovertemplate="%{y} · %{x}<br>$%{z:,.0f} wagered per available hour<extra></extra>"))
fig.update_layout(title="Demand by zone (rows) and game (columns), weakest top-left", xaxis_tickangle=-35,
                  yaxis_autorange="reversed")
save(fig, "demand_heatmap", height=480)

          theme  configured_rtp volatility_class  min_bet_cents  max_bet_cents
     Comet Cash            0.92             high             25            100
 Lucky Lanterns            0.92              low             25            100
    Neon Nights            0.88             high             40            200
Prairie Thunder            0.89           medium             40            200
    Desert Gold            0.90           medium             50            200
 Classic Sevens            0.93              low            100            300
    Aurora Link            0.91           medium             50            400
 Jackpot Summit            0.88        very_high            100            500
     Royal Draw            0.95              low             25            500
 Platinum Reels            0.90             high            500           2500 

zoom slider: no


saved demand_heatmap.html


### 2.4 Designed vs actual hold, by game

In [167]:
g = d.groupby("theme").agg(
    machines=("machine_id", "nunique"),
    coin_in=("coin_in_usd", "sum"),
    house_win=("house_win_usd", "sum"),
    theo=("theo_usd", "sum"),
    rtp=("configured_rtp", "mean"),
    volatility=("volatility_class", "first"),
)
g["actual_hold"]   = g["house_win"] / g["coin_in"]
g["designed_hold"] = g["theo"] / g["coin_in"]          # = 1 - RTP
g["gap_pts"]       = (g["actual_hold"] - g["designed_hold"]) * 100
g["win_per_machine"] = g["house_win"] / g["machines"]

print(g.sort_values("gap_pts")[["machines", "rtp", "volatility", "designed_hold",
                                "actual_hold", "gap_pts", "win_per_machine"]].round(3))

                 machines   rtp volatility  designed_hold  actual_hold  gap_pts  win_per_machine
theme                                                                                           
Comet Cash             24  0.92       high           0.08        0.077   -0.311        51125.583
Royal Draw             25  0.95        low           0.05        0.049   -0.119        40703.480
Desert Gold            48  0.90     medium           0.10        0.099   -0.102        71853.969
Lucky Lanterns         24  0.92        low           0.08        0.080   -0.028        58511.771
Prairie Thunder        66  0.89     medium           0.11        0.110   -0.025        90165.291
Classic Sevens         31  0.93        low           0.07        0.070    0.003        79793.935
Aurora Link             8  0.91     medium           0.09        0.091    0.056        67360.938
Neon Nights            42  0.88       high           0.12        0.121    0.136        93055.938
Jackpot Summit         18  0.8

### 2.5 Exposure and availability: why Aurora Link looks low

In [168]:
machines = load("data/2_machine_decision/machines.parquet")
cal = load("data/2_machine_decision/calendar_day.parquet")

dd = d.merge(cal[["gaming_date", "scheduled_seconds"]], on="gaming_date")
ex = dd.groupby("theme").agg(
    machines=("machine_id", "nunique"),
    days_per_machine=("gaming_date", "count"),
    avail_hrs=("available_hours", "sum"),
    sched_hrs=("scheduled_seconds", lambda s: s.sum() / 3600),
    faults=("faults_opened", "sum"),
)
ex["days_per_machine"] = ex["days_per_machine"] / ex["machines"]
ex["availability"] = ex["avail_hrs"] / ex["sched_hrs"]
ex["faults_per_machine"] = ex["faults"] / ex["machines"]
print(ex[["machines", "days_per_machine", "availability", "faults_per_machine"]].round(3), "\n")

# Aurora Link machines individually
al = machines[machines["theme"] == "Aurora Link"]
print(al[["machine_id", "manufacturer", "install_date", "retire_date"]].to_string(index=False))

                 machines  days_per_machine  availability  faults_per_machine
theme                                                                        
Aurora Link             8            99.000         0.995               1.750
Classic Sevens         31           179.226         0.997               3.677
Comet Cash             24           181.000         0.995               5.083
Desert Gold            48           181.000         0.996               3.979
Jackpot Summit         18           181.000         0.997               3.167
Lucky Lanterns         24           181.000         0.995               5.125
Neon Nights            42           181.000         0.996               4.000
Platinum Reels         24           175.750         0.999               0.625
Prairie Thunder        66           181.000         0.996               4.288
Royal Draw             25           173.760         0.995               4.800 

machine_id manufacturer install_date retire_date
    M-2642   

### 2.6 Replacements, additions and retirements

In [169]:
placements = load("data/2_machine_decision/placements.parquet")

# Machines that left the floor during the period
retired = machines[machines["retire_date"].notna()]
print("Retired machines:")
print(retired[["machine_id", "manufacturer", "theme", "install_date", "retire_date"]].to_string(index=False), "\n")

# Which slots the Aurora Link machines went into, and who was there before
al_ids = machines.loc[machines["theme"] == "Aurora Link", "machine_id"]
al_slots = placements.loc[placements["machine_id"].isin(al_ids), "location_id"].unique()
history = placements[placements["location_id"].isin(al_slots)] \
            .merge(machines[["machine_id", "theme"]], on="machine_id") \
            .sort_values(["location_id", "effective_from_utc"])
print("History of the Aurora Link slots:")
print(history[["location_id", "machine_id", "theme", "effective_from_utc", "effective_to_utc"]].to_string(index=False))

Retired machines:
machine_id manufacturer          theme install_date retire_date
    M-1118    Ridgeline Platinum Reels   2019-02-16  2025-02-25
    M-1120    Ridgeline     Royal Draw   2018-02-16  2025-05-13 

History of the Aurora Link slots:
location_id machine_id       theme  effective_from_utc effective_to_utc
     L-51-1     M-2642 Aurora Link 2025-03-24 13:00:00              NaT
     L-51-2     M-8743 Aurora Link 2025-03-24 13:00:00              NaT
     L-51-3     M-8057 Aurora Link 2025-03-24 13:00:00              NaT
     L-51-4     M-2799 Aurora Link 2025-03-24 13:00:00              NaT
     L-51-5     M-7943 Aurora Link 2025-03-24 13:00:00              NaT
     L-51-6     M-3101 Aurora Link 2025-03-24 13:00:00              NaT
     L-51-7     M-7756 Aurora Link 2025-03-24 13:00:00              NaT
     L-51-8     M-7487 Aurora Link 2025-03-24 13:00:00              NaT


In [170]:
# 1) Every machine installed during the six months
new = machines[machines["install_date"] >= pd.Timestamp("2025-01-01").date()]
print("Installed during the period:")
print(new[["machine_id", "manufacturer", "theme", "install_date"]].to_string(index=False), "\n")

# 2) Did the new Aurora Link bank pull play from other Center Pit machines?
cp = d[(d["zone_name"] == "Center Pit") & (~d["theme"].eq("Aurora Link"))].copy()
cp["period"] = (cp["gaming_date"] >= pd.Timestamp("2025-03-24").date()).map({False: "before Mar 24", True: "after Mar 24"})
rest = d[d["zone_name"] != "Center Pit"].copy()
rest["period"] = (rest["gaming_date"] >= pd.Timestamp("2025-03-24").date()).map({False: "before Mar 24", True: "after Mar 24"})

def rate(x):
    return x.groupby("period").apply(lambda g: g["coin_in_usd"].sum() / g["available_hours"].sum())

out = pd.DataFrame({"Center Pit (other games)": rate(cp), "Rest of floor": rate(rest)})
out.loc["change"] = out.loc["after Mar 24"] / out.loc["before Mar 24"] - 1
print("Coin-in per available hour:")
print(out.round(3))

Installed during the period:
machine_id manufacturer          theme install_date
    M-1532    Northwind Classic Sevens   2025-02-25
    M-2642      Halcyon    Aurora Link   2025-03-24
    M-2799      Halcyon    Aurora Link   2025-03-24
    M-3101      Halcyon    Aurora Link   2025-03-24
    M-5076      Halcyon     Royal Draw   2025-05-13
    M-7487      Halcyon    Aurora Link   2025-03-24
    M-7756      Halcyon    Aurora Link   2025-03-24
    M-7943      Halcyon    Aurora Link   2025-03-24
    M-8057      Halcyon    Aurora Link   2025-03-24
    M-8743      Halcyon    Aurora Link   2025-03-24 

Coin-in per available hour:
               Center Pit (other games)  Rest of floor
period                                                
after Mar 24                    207.743        173.265
before Mar 24                   219.396        186.024
change                           -0.053         -0.069


### 2.7 Natural experiment 1: the two machine replacements

In [171]:
cases = [("M-1118", "M-1532", "2025-02-25"),   # Platinum Reels -> Classic Sevens
         ("M-1120", "M-5076", "2025-05-13")]   # old Royal Draw -> new Royal Draw

for old, new_id, swap in cases:
    swap = pd.Timestamp(swap).date()
    slot = d.loc[d["machine_id"] == new_id, "location_id"].iloc[0]
    zone = d.loc[d["machine_id"] == new_id, "zone_name"].iloc[0]

    s = d[d["location_id"] == slot]                               # the slot itself
    z = d[(d["zone_name"] == zone) & (d["location_id"] != slot)]  # rest of its zone = control

    def per_hr(x, col):
        b = x[x["gaming_date"] < swap]
        a = x[x["gaming_date"] >= swap]
        return (b[col].sum() / b["available_hours"].sum(),
                a[col].sum() / a["available_hours"].sum())

    sb, sa = per_hr(s, "coin_in_usd")
    zb, za = per_hr(z, "coin_in_usd")
    tb, ta = per_hr(s, "theo_usd")

    print(f"\n{old} -> {new_id}  (slot {slot}, {zone}, swap {swap})")
    print(f"  Slot coin-in/hr:  before ${sb:,.0f}  after ${sa:,.0f}  change {sa/sb-1:+.0%}")
    print(f"  Zone coin-in/hr:  before ${zb:,.0f}  after ${za:,.0f}  change {za/zb-1:+.0%}   (control)")
    print(f"  Slot theo/hr:     before ${tb:,.2f}  after ${ta:,.2f}  change {ta/tb-1:+.0%}")


M-1118 -> M-1532  (slot L-27-3, High-Limit Room, swap 2025-02-25)
  Slot coin-in/hr:  before $193  after $220  change +14%
  Zone coin-in/hr:  before $175  after $163  change -7%   (control)
  Slot theo/hr:     before $19.28  after $15.39  change -20%

M-1120 -> M-5076  (slot L-38-5, Back Forty, swap 2025-05-13)
  Slot coin-in/hr:  before $120  after $128  change +6%
  Zone coin-in/hr:  before $122  after $123  change +1%   (control)
  Slot theo/hr:     before $6.00  after $6.39  change +6%


Swap 1 (Platinum Reels → Classic Sevens): play +14% vs the zone's −7%, but **theo/hr −20%** because the new game keeps 7% instead of 10%.
Swap 2 (old → new Royal Draw in Back Forty): about +5% vs the zone, only **+$0.39 theo/hr**, which doesn't pay for a replacement.

### 2.8 Luck test: is any single machine paying out abnormally?

In [172]:
md = load("data/2_machine_decision/machine_day.parquet")
gm = load("data/2_machine_decision/game_math.parquet")

# How much luck is normal for each machine (from bet sizes and game swinginess)
mdx = md[md["active"]].merge(gm[["config_id", "payout_multiplier_variance"]], on="config_id")
mdx["luck_var"] = mdx["sum_bet_squared"].astype(float) * mdx["payout_multiplier_variance"]
luck_sd = (mdx.groupby("machine_id")["luck_var"].sum() ** 0.5 / 100).rename("luck_sd")

# Actual vs expected per machine
lk = d.groupby("machine_id").agg(
    house_win=("house_win_usd", "sum"),
    theo=("theo_usd", "sum"),
    coin_in=("coin_in_usd", "sum"),
    rtp=("configured_rtp", "first"),
    theme=("theme", "first"),
    zone=("zone_name", "last"),
    volatility=("volatility_class", "first"),
).join(luck_sd)

lk["luck"]          = lk["house_win"] - lk["theo"]
lk["z"]             = lk["luck"] / lk["luck_sd"]
lk["actual_hold"]   = lk["house_win"] / lk["coin_in"]
lk["designed_hold"] = 1 - lk["rtp"]

# Check the luck math works: std should be near 1
print("z across all machines: mean", round(lk["z"].mean(), 2), " std", round(lk["z"].std(), 2), "\n")

cols = ["theme", "zone", "volatility", "designed_hold", "actual_hold", "luck", "z"]
print("Most unusual LOW results (possible leaks):")
print(lk.nsmallest(8, "z")[cols].round(3), "\n")
print("Most unusual HIGH results (very lucky):")
print(lk.nlargest(5, "z")[cols].round(3))

z across all machines: mean -0.07  std 1.09 

Most unusual LOW results (possible leaks):
                      theme             zone volatility  designed_hold  actual_hold       luck      z
machine_id                                                                                           
M-1158          Desert Gold       Back Forty     medium           0.10        0.030 -36164.750 -4.570
M-1054      Prairie Thunder  Garage Entrance     medium           0.11        0.090 -14528.384 -2.776
M-1210           Royal Draw    Main Entrance        low           0.05        0.036 -14740.275 -2.505
M-1010          Desert Gold       Center Pit     medium           0.10        0.071 -24632.300 -2.412
M-1181           Comet Cash       Back Forty       high           0.08        0.066  -5539.160 -2.406
M-1201      Prairie Thunder   Hotel Corridor     medium           0.11        0.093 -13115.764 -2.329
M-1288       Jackpot Summit   Hotel Corridor  very_high           0.12        0.054 -59133.080 

In [173]:
# Running total of actual − expected for every machine
dd = d.sort_values("gaming_date").copy()
dd["luck"] = dd["house_win_usd"] - dd["theo_usd"]
cum = dd.pivot_table(index="gaming_date", columns="machine_id", values="luck", aggfunc="sum").fillna(0).cumsum()
cum_z = cum / lk["luck_sd"]            # in units of each machine's normal luck (uses lk from the leak test)

highlight = {"M-1158": RED,       # leak candidate (z -4.6)
             "M-1288": ORANGE}    # biggest dollar loss, but normal for its volatility (z -2.3)

def running_luck(data, ylabel, title, name, band=False):
    fig = go.Figure()
    xs, ys = grey_lines(data, exclude=highlight)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color=FAINT, width=0.7), name="All other machines",
                    hoverinfo="skip")
    for mid, col in highlight.items():
        fig.add_scatter(x=data.index, y=data[mid], mode="lines", line=dict(color=col, width=3),
                        name=f"{mid} ({lk.loc[mid, 'theme']}, {lk.loc[mid, 'zone']})")
    if band:
        fig.add_hrect(y0=-2, y1=2, fillcolor=GREEN, opacity=0.12, line_width=0,
                      annotation_text="Normal luck range (±2)", annotation_position="top left")
    fig.add_hline(y=0, line_dash="dash", line_color=WHITE, line_width=1)
    fig.update_layout(title=f"Running total of actual − expected win: {title}", yaxis_title=ylabel,
                      xaxis_title="Date", legend=dict(orientation="h", y=-0.2))
    save(fig, name, height=440)

running_luck(cum, "USD", "in dollars", "running_luck_dollars")
running_luck(cum_z, "Luck units (machine's own SD)", "in luck units (fair comparison)", "running_luck_z", band=True)

zoom slider: yes


saved running_luck_dollars.html
zoom slider: yes


saved running_luck_z.html


The luck model is calibrated: z across all machines has mean −0.07 and std 1.09, close to the ideal 0 and 1.
**M-1158** (Desert Gold, Back Forty) kept 3% instead of 10% (z = −4.6, about 2 in a million by luck) and drifts down steadily. That makes it a **lead to inspect, not proof of wrongdoing**. M-1288 lost the most dollars ($59K) but is within normal luck for a very-high-volatility game.

### 2.9 Regression: location vs game vs machine
Target: log of coin-in per working hour, one row per active machine-day, weighted by working hours, with a term for every calendar day (so weekends and events don't bias the comparison). Needs `statsmodels`.

In [174]:
import warnings
import numpy as np
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore", message=".*rank-deficient.*")
 
daily = load("data/1_prepared_views/prepared_daily_view.parquet")
r = daily[daily["active"] & (daily["available_hours"] > 0) & (daily["coin_in_usd"] > 0)].copy()
r["y"] = np.log(r["coin_in_usd"] / r["available_hours"])     # log of $ wagered per working hour
r["date"] = r["gaming_date"].astype(str)
 
# Model 1: zone + game + each calendar day. Weighted by working hours.
m1 = smf.wls("y ~ C(zone_name, Treatment('Main Entrance')) + C(theme, Treatment('Classic Sevens')) + C(date)",
             data=r, weights=r["available_hours"]).fit()
print(f"Rows: {len(r):,}   R²: {m1.rsquared:.3f}")
 
def effects(prefix, label):
    p, ci = m1.params.filter(like=prefix), m1.conf_int().filter(like=prefix, axis=0)
    out = pd.DataFrame({"effect": np.exp(p) - 1, "low": np.exp(ci[0]) - 1, "high": np.exp(ci[1]) - 1})
    out.index = [i.split("[T.")[1].rstrip("]") for i in out.index]
    return out.sort_values("effect")
 
zone_eff = effects("zone_name", "zone")
game_eff = effects("theme", "game")
 
for eff, ref, title, name in [(zone_eff, "Main Entrance", "Zone effect", "zone_effect"),
                              (game_eff, "Classic Sevens", "Game effect", "game_effect")]:
    colors = [RED if v < -0.3 else ORANGE if v < -0.15 else GREY for v in eff["effect"]]
    fig = go.Figure(go.Bar(x=eff["effect"] * 100, y=eff.index, orientation="h", marker_color=colors,
                           error_x=dict(type="data", symmetric=False, array=(eff["high"] - eff["effect"]) * 100,
                                        arrayminus=(eff["effect"] - eff["low"]) * 100, color=WHITE, thickness=1),
                           hovertemplate="%{y}: %{x:+.1f}%<extra></extra>"))
    fig.add_vline(x=0, line_color=WHITE, line_width=1)
    fig.update_layout(title=f"{title} on play per working hour (vs {ref})", xaxis_ticksuffix="%",
                      xaxis_title="% difference, same day and other factor held equal (bars = 95% CI)")
    save(fig, name, height=400)
 
print("\nZone effects:\n", (zone_eff["effect"] * 100).round(1).astype(str) + "%")
print("\nGame effects:\n", (game_eff["effect"] * 100).round(1).astype(str) + "%")
 

Rows: 55,075   R²: 0.521
zoom slider: no


saved zone_effect.html
zoom slider: no


saved game_effect.html

Zone effects:
 Back Forty         -48.1%
High-Limit Room    -29.4%
Garage Entrance    -20.8%
Hotel Corridor     -16.0%
Bar Row            -14.9%
Center Pit          -4.1%
Name: effect, dtype: str

Game effects:
 Comet Cash         -40.9%
Platinum Reels     -40.4%
Desert Gold        -35.1%
Lucky Lanterns     -35.0%
Neon Nights        -33.0%
Jackpot Summit     -28.1%
Prairie Thunder    -26.1%
Royal Draw         -24.1%
Aurora Link         10.5%
Name: effect, dtype: str


### 2.10 Effects vs the floor average: occupancy vs bet intensity

In [175]:
# Zone and game effects vs FLOOR AVERAGE, on four normalized targets
rr = r[r["occupied_hours"] > 0].copy()
rr["y_coin"] = np.log(rr["coin_in_usd"] / rr["available_hours"])     # $ wagered per working hour
rr["y_theo"] = np.log(rr["theo_usd"] / rr["available_hours"])        # expected win per working hour
rr["y_occ"]  = np.log(rr["occupied_hours"] / rr["available_hours"])  # share of working time in use
rr["y_rate"] = np.log(rr["coin_in_usd"] / rr["occupied_hours"])      # $ wagered per hour someone is playing

def normalized_effects(factor):
    share = rr.groupby(factor)["available_hours"].sum()
    share = share / share.sum()
    res = {}
    for col, label in [("y_coin", "Coin-in / hr"), ("y_theo", "Theo / hr"),
                       ("y_occ", "Occupancy"), ("y_rate", "Coin-in per occupied hr")]:
        m = smf.wls(f"{col} ~ C(zone_name) + C(theme) + C(date)", data=rr, weights=rr["available_hours"]).fit()
        p = m.params.filter(like=factor)
        p.index = [i.split("[T.")[1].rstrip("]") for i in p.index]
        base = [z for z in share.index if z not in p.index][0]
        p[base] = 0.0
        res[label] = (np.exp(p - (p * share).sum()) - 1) * 100
    return pd.DataFrame(res).sort_values("Coin-in / hr")

zone_norm = normalized_effects("zone_name")
game_norm = normalized_effects("theme")
print("Zone effect vs floor average (%):\n", zone_norm.round(1))
print("\nGame effect vs floor average (%):\n", game_norm.round(1))

for tbl, title, name, yrange in [
        (zone_norm, "Zones differ in how often seats are used, not how much people bet", "demand_decomposition", None),
        (game_norm, "Games differ in how much seated players bet", "game_occupancy_vs_bet", [-100, 150])]:
    fig = go.Figure([
        go.Bar(x=tbl.index, y=tbl["Occupancy"], name="Occupancy (are seats used?)", marker_color=CYAN,
               hovertemplate="%{x}: occupancy %{y:+.1f}% vs floor<extra></extra>"),
        go.Bar(x=tbl.index, y=tbl["Coin-in per occupied hr"], name="Bet intensity (when seated)", marker_color=GOLD,
               hovertemplate="%{x}: bet intensity %{y:+.1f}% vs floor<extra></extra>"),
    ])
    fig.add_hline(y=0, line_color=WHITE, line_width=1)
    fig.update_layout(barmode="group", title=title, yaxis_title="% vs floor average", yaxis_ticksuffix="%",
                      xaxis_tickangle=-30, legend=dict(orientation="h", y=1.1))
    if yrange:
        fig.update_yaxes(range=yrange)   # Platinum Reels bet intensity (+439%) runs off the top; hover shows it
    save(fig, name, height=430)

Zone effect vs floor average (%):
                  Coin-in / hr  Theo / hr  Occupancy  Coin-in per occupied hr
Back Forty              -35.8      -35.8      -35.1                     -1.1
High-Limit Room         -12.7      -12.7      -14.5                      2.2
Garage Entrance          -1.9       -1.9        2.5                     -4.3
Hotel Corridor            4.0        4.0        0.6                      3.4
Bar Row                   5.3        5.3        8.5                     -2.9
Center Pit               18.8       18.8       18.1                      0.5
Main Entrance            23.8       23.8       19.1                      3.9

Game effect vs floor average (%):
                  Coin-in / hr  Theo / hr  Occupancy  Coin-in per occupied hr
Comet Cash              -16.6      -28.4       26.6                    -34.1
Platinum Reels          -15.9       -9.7      -84.4                    439.3
Desert Gold              -8.5       -1.8        8.3                    -15.5
Lucky

saved demand_decomposition.html
zoom slider: no


saved game_occupancy_vs_bet.html


Play per hour = **occupancy** (are seats used?) × **bet intensity** (how much seated players wager per hour).
- **Zones differ in occupancy.** Back Forty seats are empty 35% more than average, while seated players bet about the same (−1%).
- **Games differ in bet size.** Comet Cash and Lucky Lanterns have *above-average* occupancy but low bets ($1 max), so they are low-stakes games, not unpopular ones. Platinum Reels is the reverse; it only appears in the High-Limit Room, so its effect describes high-limit play overall.

### 2.11 How much of the difference between machines each factor explains

In [176]:
# How much of the difference BETWEEN machines does each factor explain?
day_only = smf.wls("y ~ C(date)", data=r, weights=r["available_hours"]).fit()
r["y_adj"] = day_only.resid                     # play level with day-to-day swings removed

mach = r.groupby("machine_id").apply(lambda g: pd.Series({
    "y": np.average(g["y_adj"], weights=g["available_hours"]),
    "zone": g["zone_name"].iloc[-1], "bank": g["bank_id"].iloc[-1], "theme": g["theme"].iloc[0]}))
mach["y"] = mach["y"].astype(float)

shares = pd.Series({name: smf.ols(f"y ~ {f}", data=mach).fit().rsquared for name, f in
          [("Zone only", "C(zone)"), ("Game only", "C(theme)"),
           ("Zone + game", "C(zone) + C(theme)"), ("Bank + game", "C(bank) + C(theme)")]})
print((shares * 100).round(1).astype(str) + "% of between-machine differences explained")

s = shares[::-1]
fig = go.Figure(go.Bar(x=s.values * 100, y=s.index, orientation="h", marker_color=CYAN,
                       text=[f"{v:.0%}" for v in s.values], textposition="outside",
                       hovertemplate="%{y}: %{x:.1f}%<extra></extra>"))
fig.update_layout(title="Location and game explain almost everything; the machine itself very little",
                  xaxis_title="% of differences in play between machines explained", xaxis_range=[0, 110],
                  xaxis_ticksuffix="%")
save(fig, "variance_explained", height=320)

Zone only      68.7% of between-machine differences explained
Game only      43.0% of between-machine differences explained
Zone + game    95.6% of between-machine differences explained
Bank + game    98.7% of between-machine differences explained
dtype: str
zoom slider: no


saved variance_explained.html


### 2.12 Machine effects and the manager's bottom 10

In [177]:
# Model 2: each machine compared with its own bank and game -> the "machine effect"
def machine_effects(data):
    m = smf.wls("y ~ C(bank_id) + C(theme) + C(date)", data=data, weights=data["available_hours"]).fit()
    g = data.assign(resid=m.resid).groupby("machine_id")
    eff = g.apply(lambda x: np.average(x["resid"], weights=x["available_hours"]))
    se = g["resid"].std() / np.sqrt(g.size())
    return pd.DataFrame({"machine_effect": np.exp(eff) - 1, "t": eff / se})

me = machine_effects(r)

# Stability check: are machine effects the same in Jan-Mar and Apr-Jun?
first  = machine_effects(r[r["date"] <  "2025-04-01"])["machine_effect"]
second = machine_effects(r[r["date"] >= "2025-04-01"])["machine_effect"]
both = pd.concat([first, second], axis=1, keys=["jan_mar", "apr_jun"]).dropna()
print(f"Machine effects: Jan-Mar vs Apr-Jun correlation = {both.corr().iloc[0, 1]:.2f}  (1 = fully stable, 0 = noise)")

weak = me[(me["machine_effect"] < -0.05) & (me["t"] < -3)].join(both)
weak = weak[(weak["jan_mar"] < -0.03) & (weak["apr_jun"] < -0.03)]      # weak in BOTH halves
info = daily[daily["active"]].groupby("machine_id").agg(zone=("zone_name", "last"), theme=("theme", "first"),
                                                       house_win=("house_win_usd", "sum"), days=("gaming_date", "count"))
print("\nMachines weaker than their own bank and game in BOTH halves:")
print(weak.join(info).sort_values("machine_effect")[["zone", "theme", "machine_effect", "jan_mar", "apr_jun", "t"]].round(3))

# The manager's bottom 10, explained
b10 = info.nsmallest(10, "house_win").join(me)
def reason(row):
    if row["days"] < 150:             return "Partial time on floor"
    if row["machine_effect"] < -0.05: return "Weak machine"
    if row["zone"] == "Back Forty":   return "Location (Back Forty)"
    return "Game / other"
b10["main reason"] = b10.apply(reason, axis=1)
print("\nManager's bottom 10:")
print(b10[["zone", "theme", "house_win", "days", "machine_effect", "main reason"]].round(3).to_string())

fig = go.Figure(go.Histogram(x=me["machine_effect"] * 100, nbinsx=40, marker_color=FAINT, name="All machines"))
for v in b10["machine_effect"]:
    fig.add_vline(x=v * 100, line_color=RED, opacity=0.6)
fig.add_scatter(x=b10["machine_effect"] * 100, y=[0] * len(b10), mode="markers", name="Manager's bottom 10",
                marker=dict(color=RED, size=11, symbol="diamond", line=dict(color=WHITE, width=1)),
                customdata=np.stack([b10.index, b10["zone"], b10["theme"], b10["main reason"]], axis=1),
                hovertemplate="%{customdata[0]} · %{customdata[2]} · %{customdata[1]}<br>"
                              "%{x:+.1f}% vs its own bank and game<br>Main reason: %{customdata[3]}<extra></extra>")
fig.add_vline(x=0, line_color=WHITE, line_width=1)
fig.update_layout(title="The manager's bottom 10 are normal machines for their spot and game",
                  xaxis_title="Machine effect: % play vs same bank and game, same day", xaxis_ticksuffix="%",
                  yaxis_title="Machines", bargap=0.05, legend=dict(orientation="h", y=1.1))
save(fig, "bottom10_machine_effect", height=420)

Machine effects: Jan-Mar vs Apr-Jun correlation = 0.29  (1 = fully stable, 0 = noise)

Machines weaker than their own bank and game in BOTH halves:
                      zone            theme  machine_effect  jan_mar  apr_jun      t
machine_id                                                                          
M-1184          Back Forty   Jackpot Summit          -0.140   -0.158   -0.115 -4.467
M-1154          Back Forty   Jackpot Summit          -0.105   -0.080   -0.121 -3.341
M-1084      Hotel Corridor   Lucky Lanterns          -0.072   -0.084   -0.065 -4.618
M-1171             Bar Row  Prairie Thunder          -0.068   -0.069   -0.066 -4.539
M-1081       Main Entrance   Lucky Lanterns          -0.058   -0.076   -0.043 -3.978

Manager's bottom 10:
                       zone           theme  house_win  days  machine_effect            main reason
machine_id                                                                                         
M-5076           Back Forty      Ro

saved bottom10_machine_effect.html


Machine effects are only weakly stable between Jan–Mar and Apr–Jun (correlation ≈ 0.29), so most machine-to-machine differences are noise. Only machines weak in **both** halves are flagged.

### 2.13 When are machines played? PCA on 24-hour occupancy patterns (needs `scikit-learn`)

In [178]:
# PCA on each machine's 24-hour occupancy pattern (needs scikit-learn)
from sklearn.decomposition import PCA

mh = pd.read_parquet("data/5_optional/machine_hour.parquet",
                     columns=["machine_id", "local_hour", "active", "available_seconds", "occupied_seconds"])
mh = mh[mh["active"] & (mh["available_seconds"] > 0)]

hours = list(range(6, 24)) + list(range(0, 6))            # gaming day order: 6 a.m. -> 5 a.m.
hr = mh.groupby(["machine_id", "local_hour"])[["occupied_seconds", "available_seconds"]].sum()
occ = (hr["occupied_seconds"] / hr["available_seconds"]).unstack()[hours]   # occupancy by hour, one row per machine
shape = occ.div(occ.mean(axis=1), axis=0)                 # SHAPE only: 1.0 = that machine's own average hour

pca = PCA().fit(shape)
print("Share of variation explained by each component:", (pca.explained_variance_ratio_[:4] * 100).round(1), "%")

zones = daily[daily["active"]].groupby("machine_id")["zone_name"].last()
scores = pd.DataFrame(pca.transform(shape)[:, :2], index=shape.index, columns=["PC1", "PC2"]).join(zones)
print("\nAverage PC1 by zone (higher = play more concentrated in the evening):")
print(scores.groupby("zone_name")["PC1"].mean().sort_values().round(2))

# Back Forty occupancy relative to the whole floor, hour by hour
mh = mh.join(zones, on="machine_id")
zh = mh.groupby(["zone_name", "local_hour"])[["occupied_seconds", "available_seconds"]].sum()
zone_occ = (zh["occupied_seconds"] / zh["available_seconds"]).unstack(0).reindex(hours)
fh = mh.groupby("local_hour")[["occupied_seconds", "available_seconds"]].sum().reindex(hours)
floor_occ = fh["occupied_seconds"] / fh["available_seconds"]
bf_ratio = zone_occ["Back Forty"] / floor_occ

labels = [f"{h}:00" for h in hours]

# 1) What PC1 means
fig = go.Figure(go.Scatter(x=labels, y=pca.components_[0], mode="lines+markers",
                           line=dict(color=PURPLE, width=2), marker=dict(size=5)))
fig.add_hline(y=0, line_color=WHITE, line_width=1)
fig.update_layout(title=f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variation): evening concentration",
                  yaxis_title="Loading (+ = hour gets relatively more play)", xaxis_title="Hour of the gaming day")
save(fig, "pca_evening_loading", height=380)

# 2) PC1 score by zone
order = scores.groupby("zone_name")["PC1"].mean().sort_values().index
fig = go.Figure([go.Box(x=scores.loc[scores["zone_name"] == z, "PC1"], name=z, orientation="h",
                        marker_color=RED if z == "Back Forty" else CYAN, boxpoints="all", jitter=0.4,
                        pointpos=0, marker_size=3) for z in order])
fig.add_vline(x=0, line_color=WHITE, line_width=1)
fig.update_layout(title="PC1 score by zone", showlegend=False,
                  xaxis_title="PC1 score: right = busy mostly in the evening, left = busy all day")
save(fig, "pca_by_zone", height=420)

# 3) Back Forty vs whole floor, hour by hour
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=labels, y=zone_occ["Back Forty"].values * 100, name="Back Forty occupancy",
                line=dict(color=RED, width=3), secondary_y=False)
fig.add_scatter(x=labels, y=floor_occ.values * 100, name="Whole floor occupancy",
                line=dict(color=WHITE, width=2), secondary_y=False)
fig.add_scatter(x=labels, y=bf_ratio.values * 100, name="Back Forty as % of floor (right axis)",
                line=dict(color=GOLD, dash="dash", width=2), secondary_y=True)
fig.update_xaxes(tickangle=0, tickvals=labels[::2], title_text="")      # every 2nd hour, written flat
fig.update_yaxes(title_text="Seats in use", ticksuffix="%", range=[0, 100], dtick=20, secondary_y=False)
fig.update_yaxes(title_text="Back Forty ÷ floor", ticksuffix="%", range=[0, 100], dtick=20,
                 showgrid=False, secondary_y=True)
fig.update_layout(title="Back Forty fills up when the rest of the floor is busy", hovermode="x unified",
                  legend=dict(orientation="h", x=0, y=1.02, yanchor="bottom"))
save(fig, "backforty_hourly", height=440)

print("\nBack Forty occupancy as % of floor occupancy:")
print(f"  6-9 a.m.: {bf_ratio.loc[6:9].mean():.0%}   2-4 p.m.: {bf_ratio.loc[14:16].mean():.0%}   "
      f"7-9 p.m. peak: {bf_ratio.loc[19:21].mean():.0%}   2-4 a.m.: {bf_ratio.loc[2:4].mean():.0%}")

Share of variation explained by each component: [76.5  2.8  2.5  2.1] %

Average PC1 by zone (higher = play more concentrated in the evening):
zone_name
Center Pit        -0.40
Main Entrance     -0.31
Garage Entrance   -0.22
Bar Row           -0.21
Hotel Corridor    -0.03
Back Forty         0.45
High-Limit Room    1.60
Name: PC1, dtype: float64
zoom slider: no


saved pca_evening_loading.html
zoom slider: no


saved pca_by_zone.html
zoom slider: no


saved backforty_hourly.html

Back Forty occupancy as % of floor occupancy:
  6-9 a.m.: 52%   2-4 p.m.: 69%   7-9 p.m. peak: 81%   2-4 a.m.: 55%


One component explains about 76% of the differences in daily patterns: **evening concentration**. Back Forty's occupancy is about 52% of the floor's in the morning but 81% at the 7–9 p.m. peak. It behaves like **overflow seating** that fills when the busy zones fill (a hypothesis, not a proven cause).

### 2.14 Natural experiment 2: the four machine moves

In [179]:
# Natural experiment: 4 machines moved to a new spot. Same machine, same game, different location.
placements = load("data/2_machine_decision/placements.parquet")
daily = load("data/1_prepared_views/prepared_daily_view.parquet")
d = daily[daily["active"]].copy()

counts = placements["machine_id"].value_counts()
movers = placements[placements["machine_id"].isin(counts[counts > 1].index)] \
            .sort_values(["machine_id", "effective_from_utc"])

def rates(x):
    return pd.Series({"coin_hr": x["coin_in_usd"].sum() / x["available_hours"].sum(),
                      "theo_hr": x["theo_usd"].sum() / x["available_hours"].sum(),
                      "occupancy": x["occupied_hours"].sum() / x["available_hours"].sum()})

rows = []
for mid, g in movers.groupby("machine_id"):
    move_day = d.loc[(d["machine_id"] == mid) & (d["location_id"] == g["location_id"].iloc[-1]), "gaming_date"].min()
    m = d[d["machine_id"] == mid]
    before, after = m[m["gaming_date"] < move_day], m[m["gaming_date"] > move_day]   # skip the move day itself
    old_bank, new_bank = before["bank_id"].iloc[0], after["bank_id"].iloc[0]

    # Neighbours: the other machines in the old and new bank, on the same days as each period
    nb_before = d[(d["bank_id"] == old_bank) & (d["machine_id"] != mid) & d["gaming_date"].isin(before["gaming_date"])]
    nb_after  = d[(d["bank_id"] == new_bank) & (d["machine_id"] != mid) & d["gaming_date"].isin(after["gaming_date"])]

    mb, ma, nbb, nba = rates(before), rates(after), rates(nb_before), rates(nb_after)
    rows.append({"machine": mid, "game": m["theme"].iloc[0],
                 "from": before["zone_name"].iloc[0], "to": after["zone_name"].iloc[0], "move_day": move_day,
                 "days_before": len(before), "days_after": len(after),
                 "coin_hr_before": mb["coin_hr"], "coin_hr_after": ma["coin_hr"],
                 "change": ma["coin_hr"] / mb["coin_hr"] - 1,
                 "occ_before": mb["occupancy"], "occ_after": ma["occupancy"],
                 "vs_neighbours_before": mb["coin_hr"] / nbb["coin_hr"],
                 "vs_neighbours_after": ma["coin_hr"] / nba["coin_hr"],
                 "theo_hr_change_usd": ma["theo_hr"] - mb["theo_hr"]})

moves = pd.DataFrame(rows).set_index("machine")
pd.set_option("display.width", 250)
print(moves.round(3).to_string())

print("\nHow to read it:")
print("  change               = the machine's own play per working hour, after vs before the move")
print("  vs_neighbours_before = machine ÷ the other machines in its OLD bank (same days); 1.0 = same as neighbours")
print("  vs_neighbours_after  = machine ÷ the other machines in its NEW bank (same days)")
print("  If location drives play, the machine jumps to its new neighbours' level, so both ratios stay near 1.0")

xl = [f"{mid}<br>{row['from']} → {row['to']}" for mid, row in moves.iterrows()]
fig = go.Figure([
    go.Bar(x=xl, y=moves["coin_hr_before"], name="Before move", marker_color=GREY,
           hovertemplate="Before: $%{y:,.0f}/hr<extra></extra>"),
    go.Bar(x=xl, y=moves["coin_hr_after"], name="After move", marker_color=CYAN,
           text=[f"{c:+.0%}" for c in moves["change"]], textposition="outside",
           hovertemplate="After: $%{y:,.0f}/hr<extra></extra>"),
])
fig.update_layout(barmode="group", title="Same machine, new spot: play per hour before and after each move",
                  yaxis_title="$ per working hour", yaxis_tickprefix="$", legend=dict(orientation="h", y=1.1))
save(fig, "machine_moves", height=430)

                   game            from              to    move_day  days_before  days_after  coin_hr_before  coin_hr_after  change  occ_before  occ_after  vs_neighbours_before  vs_neighbours_after  theo_hr_change_usd
machine                                                                                                                                                                                                                  
M-1113      Neon Nights      Center Pit      Center Pit  2025-02-18           48         132         192.688        200.974   0.043       0.435      0.445                 0.980                0.912               0.994
M-1126   Lucky Lanterns  Hotel Corridor         Bar Row  2025-05-06          125          55         154.457        152.405  -0.013       0.396      0.417                 0.960                0.857              -0.164
M-1152   Jackpot Summit      Back Forty   Main Entrance  2025-02-04           34         146         118.824        245.031   1.

saved machine_moves.html


In [180]:
# All machines on the three tests: weak machine (x), payout lead (y), downtime cost (bubble size)
cal = load("data/2_machine_decision/calendar_day.parquet")
dt3 = daily[daily["active"]].merge(cal[["gaming_date", "scheduled_seconds"]], on="gaming_date")
dt3["down_hrs"] = dt3["scheduled_seconds"] / 3600 - dt3["available_hours"]
per_m = dt3.groupby("machine_id").agg(theo=("theo_usd", "sum"), avail=("available_hours", "sum"),
                                      down=("down_hrs", "sum"), days=("gaming_date", "count"),
                                      zone=("zone_name", "last"), theme=("theme", "first"),
                                      house_win=("house_win_usd", "sum"))
per_m["lost_theo_per_day"] = per_m["down"] * per_m["theo"] / per_m["avail"] / per_m["days"]

three = per_m.join(me[["machine_effect"]]).join(lk[["z"]])     # me: machine effects (Model 2), lk: luck test
manager_10 = three.nsmallest(10, "house_win").index

weak_ids = three.index[(three["machine_effect"] < -0.05) & three.index.isin(weak.index)]   # weak in both halves
payout  = three.index[three["z"] < -3]
downtime = three.nlargest(10, "lost_theo_per_day").index

size = 15 + 400 * three["lost_theo_per_day"] / three["lost_theo_per_day"].max()
bubble = np.sqrt(size)          # matplotlib sizes are areas, Plotly sizes are diameters

# ---- Our worst 7: only problems that passed a test, ranked by expected win lost per day ----
per_day = daily[daily["active"]].groupby("machine_id").agg(hw=("house_win_usd", "sum"), th=("theo_usd", "sum"),
                                                          n=("gaming_date", "count"))
borderline = three.drop(payout).nsmallest(1, "z").index            # next-most-extreme payout result: watch
ours = pd.DataFrame(index=list(payout) + list(borderline) + list(weak_ids))
ours["problem"] = ["Payout lead" if m in payout else "Payout (borderline, watch)" if m in borderline
                   else "Weak machine" for m in ours.index]
ours["lost_per_day"] = [(per_day.loc[m, "th"] - per_day.loc[m, "hw"]) / per_day.loc[m, "n"] if "Payout" in p
                        else -three.loc[m, "machine_effect"] * per_day.loc[m, "th"] / per_day.loc[m, "n"]
                        for m, p in zip(ours.index, ours["problem"])]
ours = ours.join(three[["zone", "theme", "machine_effect", "z"]]).sort_values("lost_per_day", ascending=False)
ours["rank"] = range(1, len(ours) + 1)
ours["in manager's 10"] = ours.index.isin(manager_10)

HOVER = ("%{customdata[0]} · %{customdata[1]} · %{customdata[2]}<br>Machine effect %{x:+.1f}%<br>"
         "Luck z %{y:.2f}<br>Downtime cost $%{customdata[3]:,.2f}/day<extra></extra>")

def three_tests_figure(show_ours=True):
    fig = go.Figure()
    x_all = three["machine_effect"] * 100
    fig.add_vrect(x0=x_all.min() - 2, x1=-5, fillcolor=ORANGE, opacity=0.10, line_width=0)
    fig.add_hrect(y0=three["z"].min() - 0.5, y1=-3, fillcolor=RED, opacity=0.10, line_width=0)

    def add(ids, name, size=None, **marker):
        sub = three.loc[list(ids)]
        fig.add_scatter(x=sub["machine_effect"] * 100, y=sub["z"], mode="markers", name=name,
                        marker=dict(size=bubble[sub.index] if size is None else size, **marker),
                        customdata=np.stack([sub.index, sub["theme"], sub["zone"], sub["lost_theo_per_day"]], axis=1),
                        hovertemplate=HOVER)

    add(three.index, "All machines (bubble size = downtime cost per day)", color=FAINT, line=dict(color=GREY, width=0.5))
    add(manager_10, "Manager's bottom 10 (by total house win)", color="rgba(0,0,0,0)", line=dict(color=WHITE, width=2))
    add(downtime, "Top 10 downtime cost per day", color=PURPLE, line=dict(color=BG, width=0.5))
    if show_ours:
        add(ours.index, f"Our worst {len(ours)} (ranked by $ per day below expected win)", size=22, symbol="star", color=CYAN,
            line=dict(color=BG, width=1))
        for mid, row in ours.iterrows():
            fig.add_annotation(x=three.loc[mid, "machine_effect"] * 100, y=three.loc[mid, "z"], showarrow=False,
                               text=f"<b>#{row['rank']} {mid}</b>  ${row['lost_per_day']:,.0f}/day",
                               xanchor="left", xshift=12, yshift=-10, font=dict(size=10))
    # Red and orange dots drawn LAST so they sit on top of the stars and stay visible
    add(weak_ids, "Weak machine: below its own bank & game in both halves", size=10, color=ORANGE,
        line=dict(color=BG, width=1))
    add(payout, "Payout lead: luck z below −3", size=10, color=RED, line=dict(color=WHITE, width=1))
    if not show_ours:
        for mid in set(weak_ids) | set(payout):
            fig.add_annotation(x=three.loc[mid, "machine_effect"] * 100, y=three.loc[mid, "z"], text=mid,
                               showarrow=False, xanchor="left", xshift=8, yshift=6, font=dict(size=10))
    fig.add_vline(x=-5, line_dash="dash", line_color=ORANGE, line_width=1)
    fig.add_hline(y=-3, line_dash="dash", line_color=RED, line_width=1)
    fig.add_hline(y=0, line_color=FAINT, line_width=1)
    fig.add_vline(x=0, line_color=FAINT, line_width=1)
    fig.update_layout(title="Which machines are truly underperforming? Three tests on all 310 machines",
                      xaxis_title="Machine effect: % play vs its own bank and game (left = players avoid this machine)",
                      yaxis_title="Luck z-score (down = kept far less than designed)", xaxis_ticksuffix="%",
                      legend=dict(orientation="h", y=-0.2, font=dict(size=11)))
    return fig

save(three_tests_figure(show_ours=True), "three_tests", height=640)

print(f"Our worst {len(ours)} machines:")
print(ours[["rank", "problem", "zone", "theme", "machine_effect", "z", "lost_per_day", "in manager's 10"]].round(3).to_string())
overlap = int(ours["in manager's 10"].sum())
print(f"\nOverlap with the manager's bottom 10: {overlap} machine(s)\n")

flagged = sorted(set(weak_ids) | set(payout) | set(downtime))
tbl = three.loc[flagged, ["zone", "theme", "machine_effect", "z", "lost_theo_per_day"]].copy()
tbl["weak"] = tbl.index.isin(weak_ids); tbl["payout lead"] = tbl.index.isin(payout); tbl["top downtime"] = tbl.index.isin(downtime)
tbl["in manager's 10"] = tbl.index.isin(manager_10)
print(tbl.round(3).sort_values("machine_effect").to_string())

zoom slider: no


saved three_tests.html
Our worst 7 machines:
        rank                     problem             zone            theme  machine_effect      z  lost_per_day  in manager's 10
M-1158     1                 Payout lead       Back Forty      Desert Gold           0.028 -4.570       199.805             True
M-1054     2  Payout (borderline, watch)  Garage Entrance  Prairie Thunder           0.034 -2.776        80.267            False
M-1184     3                Weak machine       Back Forty   Jackpot Summit          -0.140 -0.761        40.630            False
M-1171     4                Weak machine          Bar Row  Prairie Thunder          -0.068  0.826        34.124            False
M-1154     5                Weak machine       Back Forty   Jackpot Summit          -0.105  0.155        33.478            False
M-1084     6                Weak machine   Hotel Corridor   Lucky Lanterns          -0.072  2.247        22.468            False
M-1081     7                Weak machine    Main Ent

**M-1152 moved from Back Forty to Main Entrance and doubled its play** (+106%, occupancy 17% → 34%, about +$15 theo per working hour). The regression's zone effects predicted all four moves closely (+93% predicted vs +106% actual for M-1152, −12% vs −14% for M-1299).
Constraint: there are **no empty slots** left in Main Entrance or Center Pit, so further moves would be swaps that mostly reshuffle play.

### Q1 answer
- **None of the manager's bottom 10 is a weak machine.** All 10 are normal for their bank and game: 7 are low because they sit in **Back Forty**, 3 because they were on the floor for **only part of the period**.
- **Location and game explain about 96%** of the differences between machines; the machine itself about 1%.
- **Back Forty is 36% below the floor average**, almost entirely from empty seats. The move of M-1152 confirms that the same machine doubles its play elsewhere.
- **Only 5 machines are truly weaker than their neighbours** in both halves (by 6–14%), led by M-1184 and M-1154.
- **One payout lead: M-1158**, to inspect rather than replace.
- **Replacing machines barely helped** in the two real swaps.

---
## 3. Q2: What changed in demand, and why?
Event lift = coin-in on event days vs a normal day with the same weekday; extra theo = the additional *expected* win (luck removed).

In [181]:
import decimal
import pandas as pd

def load(path):
    """Read a parquet file and turn exact-decimal money columns into normal numbers."""
    df = pd.read_parquet(path)
    for c in df.columns:
        if df[c].dtype == object and df[c].dropna().map(type).eq(decimal.Decimal).any():
            df[c] = df[c].astype(float)
    return df

prop   = load("data/1_prepared_views/property_day.parquet")
events = load("data/4_context/events.parquet")
print(events[["name", "event_type", "start_date", "end_date", "announced_date"]])
print()

# Normal level = average for each weekday on days with no event
no_event = prop["events_active"].isna() | (prop["events_active"].astype(str).str.strip().isin(["", "0", "None", "[]"]))
normal = prop[no_event].groupby("local_weekday")[["coin_in_usd", "theo_usd"]].mean()

# Lift for each event vs a normal day with the same weekday
rows = []
for _, e in events.iterrows():
    days = prop[(prop["gaming_date"] >= e["start_date"]) & (prop["gaming_date"] <= e["end_date"])]
    if days.empty:
        print(f"{e['name'][:40]:40} no matching days")
        continue
    base = normal.loc[days["local_weekday"]].sum()
    extra_theo = days["theo_usd"].sum() - base["theo_usd"]
    lift = days["coin_in_usd"].sum() / base["coin_in_usd"] - 1
    print(f"{e['name'][:40]:40} {len(days)} days  {lift:+.0%} coin-in   extra theo ≈ ${extra_theo:,.0f}")
    rows.append({"event": e["name"], "type": e["event_type"], "lift": lift, "extra_theo": extra_theo})

ev = pd.DataFrame(rows).sort_values("lift")
fig = go.Figure(go.Bar(x=ev["lift"] * 100, y=ev["event"], orientation="h",
                       marker_color=[GOLD if "Tournament" in n else CYAN for n in ev["event"]],
                       text=[f"{v:+.0%}" for v in ev["lift"]], textposition="outside",
                       customdata=np.stack([ev["type"], ev["extra_theo"]], axis=1),
                       hovertemplate="%{y} (%{customdata[0]})<br>Coin-in %{x:+.1f}% vs a normal same weekday<br>"
                                     "Extra theo ≈ $%{customdata[1]:,.0f}<extra></extra>"))
fig.update_layout(title="Event lift: coin-in vs a normal day with the same weekday (gold = our own tournament)",
                  xaxis_title="Lift in coin-in", xaxis_ticksuffix="%")
save(fig, "event_lift", height=480)

                                           name          event_type  start_date    end_date announced_date
0                                           CES          convention  2025-01-07  2025-01-10     2024-01-12
1                        Super Bowl LIX weekend      sports_viewing  2025-02-07  2025-02-09     2024-01-01
2  College basketball tournament opening rounds      sports_viewing  2025-03-20  2025-03-23     2024-12-01
3                                     CinemaCon          convention  2025-03-31  2025-04-03     2024-04-15
4                                      NAB Show          convention  2025-04-05  2025-04-09     2024-04-20
5                EDC Las Vegas festival weekend            festival  2025-05-16  2025-05-18     2024-05-20
6             Gilded Ace Winter Slot Tournament  property_promotion  2025-02-21  2025-02-23     2025-01-20
7           Gilded Ace Locals Appreciation Week  property_promotion  2025-06-09  2025-06-15     2025-05-26

CES                                 

saved event_lift.html


### Q2 answer
- **Demand was stable:** strong weekly rhythm, no long-term decline (June ran about 3% soft, see `model.ipynb`).
- **Sports weekends drove the biggest spikes:** Super Bowl +23%, college basketball +19%. EDC added +16%.
- **The casino's own Winter Tournament (+18%, about $83K extra theo) worked as well as a major outside event.** The week-long Locals Appreciation Week added only +4%.
- **Conventions barely moved play** (CES +1%, CinemaCon +3%, NAB +6%).
- Every event was announced well in advance, so events are a valid forecast input.

---
## 4. Q3: How much play was lost to downtime, and when?
Lost expected win = downtime hours × the machine's normal theo per working hour. Theo is used instead of actual win so luck doesn't distort it.

In [182]:
daily = load("data/1_prepared_views/prepared_daily_view.parquet")
d = daily[daily["active"]]
cal = load("data/2_machine_decision/calendar_day.parquet")
ops = load("data/2_machine_decision/operations.parquet")

dt = d.merge(cal[["gaming_date", "scheduled_seconds"]], on="gaming_date").copy()
dt["down_hrs"] = dt["scheduled_seconds"] / 3600 - dt["available_hours"]

# Each machine's normal expected win per working hour
rate = dt.groupby("machine_id").apply(lambda g: g["theo_usd"].sum() / g["available_hours"].sum()).rename("theo_per_hr")
dt = dt.join(rate, on="machine_id")
dt["lost_theo"] = dt["down_hrs"] * dt["theo_per_hr"]

print(f"Total downtime: {dt['down_hrs'].sum():,.0f} machine-hours")
print(f"Lost expected win: ${dt['lost_theo'].sum():,.0f}  "
      f"({dt['lost_theo'].sum() / dt['theo_usd'].sum():.2%} of all expected win)\n")

# 1) By manufacturer (preventive maintenance is bought per manufacturer)
by_mfr = dt.groupby("manufacturer").agg(lost=("lost_theo", "sum"), machines=("machine_id", "nunique"))
by_mfr["lost_per_machine"] = by_mfr["lost"] / by_mfr["machines"]
by_mfr = by_mfr.sort_values("lost_per_machine")
fig = go.Figure(go.Bar(x=by_mfr["lost_per_machine"], y=by_mfr.index, orientation="h",
                       marker_color=[RED if mf == "Ridgeline" else GREY for mf in by_mfr.index],
                       text=[f"${v:,.0f}" for v in by_mfr["lost_per_machine"]], textposition="outside",
                       customdata=by_mfr["machines"],
                       hovertemplate="%{y}: $%{x:,.0f} per machine (%{customdata} machines)<extra></extra>"))
fig.update_layout(title="Lost expected win per machine, by manufacturer", xaxis_title="USD per machine",
                  xaxis_tickprefix="$")
save(fig, "downtime_by_manufacturer", height=360)

# 2) By month
dt["month_name"] = pd.to_datetime(dt["gaming_date"]).dt.strftime("%m-%b")
by_month = dt.groupby("month_name")["lost_theo"].sum()
fig = go.Figure(go.Bar(x=by_month.index, y=by_month.values, marker_color=CYAN,
                       text=[f"${v / 1000:,.1f}K" for v in by_month.values], textposition="outside"))
fig.update_layout(title="Lost expected win by month: no trend", yaxis_title="USD", yaxis_tickprefix="$")
save(fig, "downtime_by_month", height=360)

# 3) Top 15 machines
top = dt.groupby("machine_id").agg(lost=("lost_theo", "sum"), zone=("zone_name", "last"),
                                   mfr=("manufacturer", "first")).nlargest(15, "lost")[::-1]
fig = go.Figure(go.Bar(x=top["lost"], y=[f"{i} ({row.mfr})" for i, row in top.iterrows()], orientation="h",
                       marker_color=[RED if mf == "Ridgeline" else GREY for mf in top["mfr"]],
                       customdata=top["zone"], hovertemplate="%{y} · %{customdata}<br>$%{x:,.0f} lost<extra></extra>"))
fig.update_layout(title="15 machines with the most lost expected win (red = Ridgeline)", xaxis_title="USD",
                  xaxis_tickprefix="$")
save(fig, "downtime_top15", height=500)

# What kinds of tickets, and how long machines were down for each
ops["hours_down"] = (pd.to_datetime(ops["available_at_utc"]) - pd.to_datetime(ops["opened_at_utc"])).dt.total_seconds() / 3600
print(ops.groupby("event_type").agg(tickets=("operation_id", "count"),
                                    median_hrs_down=("hours_down", "median"),
                                    total_hrs_down=("hours_down", "sum")).round(1))

Total downtime: 5,021 machine-hours
Lost expected win: $90,313  (0.38% of all expected win)

zoom slider: no


saved downtime_by_manufacturer.html
zoom slider: no


saved downtime_by_month.html
zoom slider: no


saved downtime_top15.html
                       tickets  median_hrs_down  total_hrs_down
event_type                                                     
fault_repair              1207              3.2          4934.1
install                     10              6.0            60.0
move                         4              3.0            12.0
retire                       2              0.0             0.0
scheduled_maintenance        3              5.0            15.0


### Q3 answer
- **$90,313 of expected win was lost**, only **0.38%** of the total. Machines were available about 99.5% of the time.
- **Ridgeline machines lose about 2–3× more per machine** than Northwind or Halcyon; 14 of the 15 worst machines are Ridgeline (the two retired machines were Ridgelines from 2018–2019).
- **No trend over time:** monthly losses ranged from about $11K to $18K.
- Assumption: a broken machine would have been played at its average rate. Breakdowns at quiet hours would mean a smaller real loss.

---
## 5. Q4: What did tracked players wager, get back and net?
Loyalty-card players only, so totals don't match machine meters (expected).

In [183]:
import numpy as np

pp = load("data/1_prepared_views/player_period.parquet")

# One row per player for the full six months
pl = pp.groupby("player_id").agg(
    coin_in=("tracked_coin_in_usd", "sum"),
    payout=("tracked_payout_usd", "sum"),
    theo=("theo_usd", "sum"),
    visits=("observed_visits", "sum"),
    tier=("tier_end", "last"),
    age_band=("age_band", "first"),
    region=("home_region", "first"),
)
pl["player_loss"] = pl["coin_in"] - pl["payout"]      # positive = player lost money

n = len(pl)
print(f"Players: {n:,}")
print(f"Total wagered: ${pl['coin_in'].sum():,.0f}   Paid back: ${pl['payout'].sum():,.0f}   "
      f"Net player loss: ${pl['player_loss'].sum():,.0f}")
print(f"Players who came out ahead: {(pl['player_loss'] < 0).mean():.1%}")
print(f"Median player result: lost ${pl['player_loss'].median():,.0f}\n")

# Concentration: what share of all losses comes from the biggest losers?
losers = pl.loc[pl["player_loss"] > 0, "player_loss"].sort_values(ascending=False)
for pct in [0.01, 0.05, 0.10, 0.20]:
    k = max(1, int(len(losers) * pct))
    print(f"Top {pct:.0%} of losing players account for {losers.iloc[:k].sum() / losers.sum():.1%} of all losses")

# 1) Distribution of results (clipped so the middle is visible)
fig = go.Figure(go.Histogram(x=pl["player_loss"].clip(-500, 2000), nbinsx=60, marker_color=CYAN))
fig.add_vline(x=0, line_dash="dash", line_color=WHITE, annotation_text="broke even")
fig.update_layout(title="Six-month result per player (clipped at −$500 / $2,000)", xaxis_tickprefix="$",
                  xaxis_title="Player loss (negative = player won)", yaxis_title="Players", bargap=0.03)
save(fig, "player_results", height=380)

# 2) Concentration curve (thinned to ~2,000 points so the page stays light)
cum = losers.cumsum() / losers.sum()
share_players = np.arange(1, len(losers) + 1) / len(losers)
step = max(1, len(cum) // 2000)
k10 = max(1, int(len(losers) * 0.10))
fig = go.Figure()
fig.add_scatter(x=share_players[::step] * 100, y=cum.values[::step] * 100, name="Actual", fill="tozeroy",
                line=dict(color=RED, width=3), fillcolor="rgba(228,87,46,0.15)",
                hovertemplate="Top %{x:.1f}% of losing players → %{y:.1f}% of losses<extra></extra>")
fig.add_scatter(x=[0, 100], y=[0, 100], name="If everyone lost equally", line=dict(color=GREY, dash="dash"),
                hoverinfo="skip")
fig.add_annotation(x=10, y=cum.iloc[k10 - 1] * 100, text=f"Top 10% → {cum.iloc[k10 - 1]:.0%} of losses",
                   showarrow=True, ax=70, ay=30, font=dict(color=GOLD), arrowcolor=GOLD)
fig.update_layout(title="How concentrated are losses?", xaxis_title="% of losing players (biggest losers first)",
                  yaxis_title="% of all player losses", xaxis_ticksuffix="%", yaxis_ticksuffix="%",
                  legend=dict(orientation="h", y=-0.2))
save(fig, "player_loss_concentration", height=420)

# 3) By loyalty tier
by_tier = pl.groupby("tier").agg(players=("player_loss", "size"), total_loss=("player_loss", "sum"))
by_tier["share_players"] = by_tier["players"] / by_tier["players"].sum() * 100
by_tier["share_loss"] = by_tier["total_loss"] / by_tier["total_loss"].sum() * 100
fig = go.Figure([
    go.Bar(x=by_tier.index, y=by_tier["share_players"], name="% of players", marker_color=CYAN,
           text=[f"{v:.0f}%" for v in by_tier["share_players"]], textposition="outside"),
    go.Bar(x=by_tier.index, y=by_tier["share_loss"], name="% of total losses", marker_color=GOLD,
           text=[f"{v:.0f}%" for v in by_tier["share_loss"]], textposition="outside"),
])
fig.update_layout(barmode="group", title="Players vs losses, by loyalty tier (end of period)",
                  yaxis_ticksuffix="%", legend=dict(orientation="h", y=1.1))
save(fig, "tier_players_vs_losses", height=400)

Players: 113,890
Total wagered: $163,804,870   Paid back: $148,219,409   Net player loss: $15,585,461
Players who came out ahead: 13.6%
Median player result: lost $72

Top 1% of losing players account for 15.7% of all losses
Top 5% of losing players account for 40.7% of all losses
Top 10% of losing players account for 52.8% of all losses
Top 20% of losing players account for 66.5% of all losses
zoom slider: no


saved player_results.html
zoom slider: no


saved player_loss_concentration.html
zoom slider: no


saved tier_players_vs_losses.html


### Q4 answer
- Card players wagered **$163.8M**, got back **$148.2M**, and lost **$15.6M** (about 9.5%).
- The **median player lost $72** over six months; **13.6% came out ahead**.
- **Losses are concentrated:** the top 10% of losing players account for **53%** of losses; Gold and Platinum are about 6% of players but about 57% of losses (tiers are earned by playing, so this is association, not cause).
- In line with the rules, we suggest only **protective** ideas for heavy losers, never ideas that push them to gamble more.

---
## 6. Q5: What did the comp program cost, and did anything change?

In [184]:
prop = load("data/1_prepared_views/property_day.parquet")
led  = load("data/3_guests/loyalty_ledger.parquet")

# Keep only rows that cost the casino something (redeemed comps)
c = led[led["economic_cost_cents"].fillna(0) > 0].copy()
c["cost_usd"] = c["economic_cost_cents"] / 100

# UTC -> Las Vegas local time -> gaming day (6 a.m. to 6 a.m.)
local = pd.to_datetime(c["event_at_utc"], utc=True).dt.tz_convert("America/Los_Angeles")
c["gaming_date"] = (local - pd.Timedelta(hours=6)).dt.date
c["tier"] = c["tier_before"]

# Check: ledger total should be close to the property total
print(f"Comp cost from property_day:   ${prop['comp_cost_usd'].sum():,.0f}")
print(f"Comp cost from loyalty_ledger: ${c['cost_usd'].sum():,.0f}\n")

# Weekly total (property_day)
p = prop.set_index(pd.to_datetime(prop["gaming_date"]))
weekly_total = p["comp_cost_usd"].resample("W").sum()

# Weekly by tier (loyalty_ledger)
by_tier = c.pivot_table(index="gaming_date", columns="tier", values="cost_usd", aggfunc="sum").fillna(0)
by_tier.index = pd.to_datetime(by_tier.index)
tier_order = [t for t in ["Base", "Gold", "Platinum"] if t in by_tier.columns] + \
             [t for t in by_tier.columns if t not in ["Base", "Gold", "Platinum"]]
weekly_tier = by_tier[tier_order].resample("W").sum()

print("Share of comp cost by tier:")
print((weekly_tier.sum() / weekly_tier.sum().sum() * 100).round(1).astype(str) + "%")

# Drop the partial first and last weeks (only a few days each), which otherwise look like a crash
wt_total = weekly_total.iloc[1:-1]
wt_tier = weekly_tier.iloc[1:-1]

# 1) Total comp cost per week
fig = go.Figure(go.Scatter(x=wt_total.index, y=wt_total.values, mode="lines+markers",
                           line=dict(color=CYAN, width=2), marker=dict(size=6),
                           hovertemplate="Week ending %{x|%b %d}: $%{y:,.0f}<extra></extra>"))
date_line(fig, "2025-05-01", "May 1: comps get cheaper in points")
fig.update_layout(yaxis_title="Comp cost per week", yaxis_tickprefix="$", margin=dict(t=50))
save(fig, "comp_weekly_total", height=400)

# 2) Comp cost per week, stacked by tier (no inner title: the dashboard card has one)
fig = go.Figure([go.Scatter(x=wt_tier.index, y=wt_tier[t], name=t, stackgroup="one", mode="lines",
                            line=dict(width=0.5, color=TIER_COLORS.get(t, CYAN)),
                            hovertemplate=f"{t}: $%{{y:,.0f}}<extra></extra>") for t in wt_tier.columns])
date_line(fig, "2025-05-01", "May 1")
fig.update_layout(yaxis_title="Comp cost per week", yaxis_tickprefix="$", hovermode="x unified",
                  margin=dict(t=60), legend=dict(orientation="h", x=0, y=1.02, yanchor="bottom", traceorder="normal"))
save(fig, "comp_weekly_by_tier", height=420)

Comp cost from property_day:   $190,669
Comp cost from loyalty_ledger: $190,669

Share of comp cost by tier:
tier
Base        38.6%
Gold        49.7%
Platinum    11.7%
dtype: str
zoom slider: yes


saved comp_weekly_total.html
zoom slider: yes


saved comp_weekly_by_tier.html


In [185]:
# Weekly: how many players redeemed in each tier, and how much each got on average
c["week"] = pd.to_datetime(c["gaming_date"]).dt.to_period("W").dt.start_time
w = c.groupby(["week", "tier"]).agg(players=("player_id", "nunique"), cost=("cost_usd", "sum")).reset_index()
w["cost_per_player"] = w["cost"] / w["players"]

for col, title, name, prefix in [("players", "Players redeeming comps per week, by tier", "comp_redeemers", ""),
                                 ("cost_per_player", "Average comp cost per redeeming player, by tier",
                                  "comp_cost_per_player", "$")]:
    fig = go.Figure([go.Scatter(x=g["week"], y=g[col], name=t, mode="lines",
                                line=dict(color=TIER_COLORS.get(t), width=2.5)) for t, g in w.groupby("tier")])
    date_line(fig, "2025-05-01", "May 1")
    fig.update_layout(title=title, yaxis_tickprefix=prefix, hovermode="x unified", legend=dict(orientation="h", y=1.1))
    save(fig, name, height=380)

zoom slider: yes


saved comp_redeemers.html
zoom slider: yes


saved comp_cost_per_player.html


In [186]:
pp = load("data/1_prepared_views/player_period.parquet")
mo = pp.groupby("period_month").agg(points=("points_earned", "sum"),
                                    coin_in=("tracked_coin_in_usd", "sum"),
                                    comp_cost=("comp_cost_usd", "sum"),
                                    theo=("theo_usd", "sum"))
mo["points_per_100_bet"] = mo["points"] / mo["coin_in"] * 100
mo["comp_cost_pct_of_theo"] = mo["comp_cost"] / mo["theo"] * 100
print(mo[["points_per_100_bet", "comp_cost_pct_of_theo"]].round(3))
#hellooo

              points_per_100_bet  comp_cost_pct_of_theo
period_month                                           
2025-01                   99.880                  1.114
2025-02                   99.886                  1.204
2025-03                   99.889                  1.217
2025-04                   99.887                  1.242
2025-05                   99.885                  1.296
2025-06                   99.886                  1.317


In [187]:
# Did the PRICE of comps in points change? Points redeemed per $1 of comp value, by gaming day
led = load("data/3_guests/loyalty_ledger.parquet")
red = led[led["event_type"] == "redeem"].copy()
local = pd.to_datetime(red["event_at_utc"], utc=True).dt.tz_convert("America/Los_Angeles")
red["gaming_date"] = pd.to_datetime((local - pd.Timedelta(hours=6)).dt.date)
red = red[red["gaming_date"] <= "2025-06-30"]
red["points_per_comp_dollar"] = -red["points_delta"] / (red["face_value_cents"] / 100)

by_day = red.groupby("gaming_date").agg(points_per_dollar=("points_per_comp_dollar", "median"),
                                        redemptions=("points_delta", "size"),
                                        face_usd=("face_value_cents", lambda s: s.sum() / 100),
                                        cost_usd=("economic_cost_cents", lambda s: s.sum() / 100))
red["period"] = np.where(red["gaming_date"] >= "2025-05-01", "May 1 onward", "Before May 1")
summary = red.groupby("period").agg(days=("gaming_date", "nunique"), redemptions=("points_delta", "size"),
                                    points=("points_delta", lambda s: -s.sum()),
                                    face_usd=("face_value_cents", lambda s: s.sum() / 100),
                                    cost_usd=("economic_cost_cents", lambda s: s.sum() / 100))
summary["points_per_comp_dollar"] = summary["points"] / summary["face_usd"]
summary["redemptions_per_day"] = summary["redemptions"] / summary["days"]
summary["comp_cost_per_day"] = summary["cost_usd"] / summary["days"]
summary["avg_comp_face_usd"] = summary["face_usd"] / summary["redemptions"]
print(summary[["points_per_comp_dollar", "redemptions_per_day", "comp_cost_per_day", "avg_comp_face_usd"]].round(2).to_string())

print("\nPoints per $1 of comp, by loyalty tier:")
print(red.pivot_table(index="tier_before", columns="period", values="points_per_comp_dollar", aggfunc="median").round(2))

fig = go.Figure(go.Scatter(x=by_day.index, y=by_day["points_per_dollar"], mode="lines",
                           line=dict(color=PURPLE, width=2),
                           hovertemplate="%{x|%b %d}: %{y:.2f} points per $1<extra></extra>"))
date_line(fig, "2025-05-01", "May 1")
fig.update_layout(title="Points needed per $1 of comp (median redemption, by day)", yaxis_title="Points per comp dollar")
save(fig, "comp_price_change", height=380)

weekly = by_day.resample("W")[["redemptions", "cost_usd"]].sum()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=weekly.index, y=weekly["redemptions"], name="Redemptions per week",
                line=dict(color=CYAN, width=2.5), secondary_y=False)
fig.add_scatter(x=weekly.index, y=weekly["cost_usd"], name="Comp cost per week ($, right axis)",
                line=dict(color=GREEN, width=2.5), secondary_y=True)
date_line(fig, "2025-05-01", "May 1")
fig.update_yaxes(title_text="Redemptions per week", secondary_y=False)
fig.update_yaxes(title_text="Comp cost per week", tickprefix="$", showgrid=False, secondary_y=True)
fig.update_layout(title="Did cheaper comps lead to more redemptions?", hovermode="x unified",
                  legend=dict(orientation="h", y=-0.2))
save(fig, "comp_redemptions", height=400)

              points_per_comp_dollar  redemptions_per_day  comp_cost_per_day  avg_comp_face_usd
period                                                                                         
Before May 1                   13.09               321.62            1027.09              11.45
May 1 onward                    5.78               340.43            1105.23              11.61

Points per $1 of comp, by loyalty tier:
period       Before May 1  May 1 onward
tier_before                            
Base                 15.0          7.00
Gold                 12.0          5.66
Platinum              9.0          4.22
zoom slider: yes


saved comp_price_change.html
zoom slider: yes


saved comp_redemptions.html


### Q5 answer
- The comp program cost **$190,669** over six months, about **1.1–1.3% of expected win**.
- **The earn rate never changed** (about 1 point per $1 wagered every month).
- **The redemption price changed on May 1.** The points needed per $1 of comp were cut by about 53% in every tier (Base 15 → 7, Gold 12 → 5.66, Platinum 9 → 4.22). This was a deliberate policy change: earn rates, comp values and the casino's cost per comp didn't move.
- **Effect of the change:** slightly more players redeemed (23% → 25%) and comp cost rose about 8%, but visits per player, coin-in per player, carded share and floor play showed **no measurable lift**. With two months of data and no control group, the effect on play is unproven.
- **Before May, the cost share rose gradually** (1.11% → 1.24% of theo) because players **moved up tiers**: Platinum redeemers grew from about 20 to 450 per week while cost per redeeming player stayed roughly flat within each tier.
- **Risk to flag:** each unredeemed point is now worth about twice as much in comps, so the potential future cost of saved points has roughly doubled.

In [188]:
# Same three-test chart without our ranking stars (built by three_tests_figure() in the Q1 section)
save(three_tests_figure(show_ours=False), "three_tests_plain", height=640)

flagged = sorted(set(weak_ids) | set(payout) | set(downtime))
tbl = three.loc[flagged, ["zone", "theme", "machine_effect", "z", "lost_theo_per_day"]].copy()
tbl["weak"] = tbl.index.isin(weak_ids); tbl["payout lead"] = tbl.index.isin(payout); tbl["top downtime"] = tbl.index.isin(downtime)
tbl["in manager's 10"] = tbl.index.isin(manager_10)
print(tbl.round(3).sort_values("machine_effect").to_string())

zoom slider: no


saved three_tests_plain.html
                       zone            theme  machine_effect      z  lost_theo_per_day   weak  payout lead  top downtime  in manager's 10
machine_id                                                                                                                               
M-1184           Back Forty   Jackpot Summit          -0.140 -0.761              1.006   True        False         False            False
M-1154           Back Forty   Jackpot Summit          -0.105  0.155              0.293   True        False         False            False
M-1084       Hotel Corridor   Lucky Lanterns          -0.072  2.247              0.662   True        False         False            False
M-1171              Bar Row  Prairie Thunder          -0.068  0.826              1.696   True        False         False            False
M-1081        Main Entrance   Lucky Lanterns          -0.058 -1.908              2.862   True        False         False            False
M-129

---
## 7. Q6: Which actions would we fund with $100,000, and how would we know they worked?

Rules: July 1–30, 2025; up to 3 action types from `action_catalog`; keeping money is valid. Budget = qty × (unit cost + daily cost × max(0, 30 − lead days)). Benefits count only for the days left after each action's lead time and are stated as **low / central / high** scenarios. Benefit = expected win (theo, so luck is excluded) + food-and-drink margin − extra comp cost.

### Q6 answer: fund 3 actions for $23,700 and keep $76,300

| Action | Cost | Expected July benefit (low – central – high) | Key assumption | How we'd measure it |
|---|---|---|---|---|
| **Inspect M-1158** | $1,200 (1-day lead) | $0 – $2.9K – $5.8K (about $73K/year if fixed) | 50% chance a fixable payout fault is found | Inspection finding; July actual hold vs designed 10% |
| **Signs for Back Forty** | $7,500 (10-day lead, 20 days of effect) | $0 – $14.8K – $29.6K | Closes 10% (central) to 20% (high) of Back Forty's gap; **break-even at +2.8% play** | Back Forty occupancy vs other zones (especially daytime, now 52–69% of floor); whole floor vs forecast to catch cannibalization |
| **One tournament-style promotion weekend, Jul 18–20** | $15,000 (14-day lead) | $34.6K – $60.6K – $86.5K | Summer lift 40–100% of the Winter Tournament's $86.5K net | Coin-in on Jul 18–20 above the forecast's **P90 line** (`model.ipynb`); lift in theo, comps and F&B vs normal same weekdays |
| **Keep** | $76,300 | — | — | — |

**Rejected:** replacing the bottom 10 or the 5 weak machines (a $3,380 lease recovers at most about $650 in July), moving machines (no empty slots in busy zones, so moves reshuffle play), an extra technician (all downtime ≈ $500/day; machines barely wait for repairs), Ridgeline preventive maintenance (about $3K benefit for $9K), and a floor-wide comp change (no evidence on player response).

**Next in line (August):** a **comp pilot with a control group ($4,000)** to test whether the May 1 halving of the comp price changes play. It costs about $28K a year in extra comps and doubled the value of unredeemed points. It was left out of July only because the guide allows 3 action types.

**Watch list (no spend):** M-1054 and M-1210. Recalculate their luck z after July and inspect only if it falls below −3.

**Responsible play:** the promotion is a broad property event, with no targeted invitations to heavy losers.

In [189]:
# The $100,000 menu: action costs from action_catalog, converted to dollars and July budget  (self-contained cell)
import decimal
import pandas as pd

catalog = pd.read_parquet("data/2_machine_decision/action_catalog.parquet")
for c in catalog.columns:
    if catalog[c].dtype == object and catalog[c].dropna().map(type).eq(decimal.Decimal).any():
        catalog[c] = catalog[c].astype(float)
catalog = catalog.set_index("action_type_id")

menu = pd.DataFrame({
    "action": catalog["name"],
    "unit_cost_$": catalog["unit_cost_cents"] / 100,
    "daily_cost_$": catalog["recurring_cost_cents_per_day"] / 100,
    "lead_days": catalog["lead_days"],
    "days_of_effect_in_july": (30 - catalog["lead_days"]).clip(lower=0),
    "downtime_min": catalog["downtime_minutes"],
    "max_qty": catalog["quantity_limit"],
    "conflicts_with": catalog["incompatibility_group"].fillna(""),
})

# Guide formula for one unit: unit cost + daily cost x max(0, 30 - lead days)
menu["july_cost_per_unit_$"] = menu["unit_cost_$"] + menu["daily_cost_$"] * menu["days_of_effect_in_july"]

pd.set_option("display.width", 250)
print(menu.to_string())
print("\nNotes from the catalog:")
for aid, note in catalog["notes"].items():
    print(f"  {aid:16} {note}")

                                                            action  unit_cost_$  daily_cost_$  lead_days  days_of_effect_in_july  downtime_min  max_qty conflicts_with  july_cost_per_unit_$
action_type_id                                                                                                                                                                              
A-MOVE-MACHINE                                Relocate one machine       1800.0           0.0          2                      28           180       24   same_machine                1800.0
A-MOVE-BANK                                  Relocate a whole bank       9500.0           0.0          5                      25           480        3   same_machine                9500.0
A-REPLACE-LEASE          Replace a machine with a leased new title       2500.0          55.0         14                      16           360       20   same_machine                3380.0
A-REPLACE-BUY         Replace a machine with a purchase

In [190]:
# Inspection candidates vs their peers: is each result too extreme to be luck?
# (self-contained except for the Plotly setup cell at the top: colors and save())
import decimal
from math import erf, sqrt
import numpy as np
import pandas as pd

def load_num(path):
    """Read a parquet file and turn exact-decimal money columns into normal numbers."""
    df = pd.read_parquet(path)
    for c in df.columns:
        if df[c].dtype == object and df[c].dropna().map(type).eq(decimal.Decimal).any():
            df[c] = df[c].astype(float)
    return df

cands = {"M-1158": RED, "M-1054": ORANGE, "M-1210": PURPLE}

# Luck model inputs per machine and per day: expected luck variance = sum_bet_squared x payout variance
pv  = load_num("data/1_prepared_views/prepared_daily_view.parquet")
md  = load_num("data/2_machine_decision/machine_day.parquet")
gmx = load_num("data/2_machine_decision/game_math.parquet")
mdx = md[md["active"]].merge(gmx[["config_id", "payout_multiplier_variance"]], on="config_id")
mdx["luck_var"] = mdx["sum_bet_squared"].astype(float) * mdx["payout_multiplier_variance"].astype(float) / 1e4   # dollars²
dd = pv[pv["active"]].merge(mdx[["machine_id", "gaming_date", "luck_var"]], on=["machine_id", "gaming_date"])
dd["luck"] = dd["house_win_usd"] - dd["theo_usd"]

info = dd.groupby("machine_id").agg(theme=("theme", "first"), zone=("zone_name", "last"),
                                    coin=("coin_in_usd", "sum"), hw=("house_win_usd", "sum"), theo=("theo_usd", "sum"),
                                    luck=("luck", "sum"), var=("luck_var", "sum"))
info["z"] = info["luck"] / np.sqrt(info["var"])
info["actual_hold"] = info["hw"] / info["coin"]
info["designed_hold"] = info["theo"] / info["coin"]

# How many machines this extreme would we expect by pure chance among all machines?
n = len(info)
norm_cdf = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
print(f"Luck model check: z across all machines has mean {info['z'].mean():+.2f} and std {info['z'].std():.2f} (ideal 0 and 1)\n")
print(f"{'machine':8} {'game':16} {'zone':15} {'designed':>8} {'actual':>7} {'z':>6}  {'expected this low by chance':>28}  {'actually this low':>17}")
for mid in cands:
    r = info.loc[mid]
    print(f"{mid:8} {r['theme']:16} {r['zone']:15} {r['designed_hold']:8.1%} {r['actual_hold']:7.1%} {r['z']:6.2f}  "
          f"{n * norm_cdf(r['z']):28.2f}  {(info['z'] <= r['z']).sum():17d}")

# 1) Distribution of luck z for all machines vs what pure luck predicts
counts, edges = np.histogram(info["z"], bins=35)
binw = edges[1] - edges[0]
fig = go.Figure(go.Bar(x=(edges[:-1] + edges[1:]) / 2, y=counts, width=binw, name="All machines",
                       marker=dict(color=FAINT, line=dict(color=GREY, width=0.5))))
xs = np.linspace(-5, 5, 200)
fig.add_scatter(x=xs, y=n * binw * np.exp(-xs**2 / 2) / np.sqrt(2 * np.pi), name="Expected from pure luck",
                line=dict(color=WHITE, width=2))
for mid, col in cands.items():
    fig.add_vline(x=info.loc[mid, "z"], line_color=col, line_width=3)
    fig.add_scatter(x=[None], y=[None], mode="lines", line=dict(color=col, width=3),
                    name=f"{mid} (z = {info.loc[mid, 'z']:.2f})")
fig.update_layout(title="Luck z-score, all machines", xaxis_title="z (below −3 ≈ too extreme for luck)",
                  yaxis_title="Machines", bargap=0)
save(fig, "luck_z_distribution", height=400)

# 2) Actual hold vs every other machine running the same game
games = [info.loc[m, "theme"] for m in cands]
fig = go.Figure()
for i, (g, mid) in enumerate(zip(games, cands)):
    peers = info[(info["theme"] == g) & (info.index != mid)]
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(peers))
    fig.add_scatter(x=i + jitter, y=peers["actual_hold"] * 100, mode="markers", name="Other machines, same game",
                    legendgroup="peers", showlegend=(i == 0), text=peers.index,
                    marker=dict(color=FAINT, size=8, line=dict(color=GREY, width=0.5)),
                    hovertemplate="%{text}: %{y:.1f}% hold<extra></extra>")
    fig.add_scatter(x=[i - 0.3, i + 0.3], y=[peers["designed_hold"].iloc[0] * 100] * 2, mode="lines",
                    name="Designed hold (1 − RTP)", legendgroup="design", showlegend=(i == 0),
                    line=dict(color=WHITE, dash="dash"), hoverinfo="skip")
for i, (mid, col) in enumerate(cands.items()):
    fig.add_scatter(x=[i], y=[info.loc[mid, "actual_hold"] * 100], mode="markers", name=mid,
                    marker=dict(color=col, size=16, line=dict(color=BG, width=1)),
                    hovertemplate=f"{mid}: %{{y:.1f}}% hold<extra></extra>")
fig.update_xaxes(tickvals=list(range(len(games))), ticktext=[f"{g}<br>({m})" for g, m in zip(games, cands)])
fig.update_layout(title="Each candidate vs all machines running the same game",
                  yaxis_title="Actual hold over 6 months", yaxis_ticksuffix="%")
save(fig, "inspection_candidates_hold", height=430)

# 3) Running luck (in luck units) vs same-game peers, one panel per candidate
dd = dd.sort_values("gaming_date")
fig = make_subplots(rows=1, cols=len(cands), horizontal_spacing=0.06,
                    subplot_titles=[f"{mid} vs other {info.loc[mid, 'theme']} machines" for mid in cands])
for i, (mid, col) in enumerate(cands.items(), start=1):
    g = info.loc[mid, "theme"]
    sub = dd[dd["theme"] == g]
    cl = sub.pivot_table(index="gaming_date", columns="machine_id", values="luck", aggfunc="sum").fillna(0).cumsum()
    cv = sub.pivot_table(index="gaming_date", columns="machine_id", values="luck_var", aggfunc="sum").fillna(0).sum()
    cz = cl / np.sqrt(cv)
    gx, gy = grey_lines(cz, exclude=[mid])
    fig.add_scatter(x=gx, y=gy, mode="lines", line=dict(color=FAINT, width=0.7), name="Same-game machines",
                    legendgroup="peers", showlegend=(i == 1), hoverinfo="skip", row=1, col=i)
    fig.add_scatter(x=cz.index, y=cz[mid], mode="lines", line=dict(color=col, width=3), name=mid, row=1, col=i)
    fig.add_hrect(y0=-2, y1=2, fillcolor=GREEN, opacity=0.12, line_width=0, row=1, col=i)
    fig.add_hline(y=0, line_color=FAINT, line_width=1, row=1, col=i)
fig.update_yaxes(title_text="Cumulative luck (final SD units)", row=1, col=1)
fig.update_layout(title="Running luck vs same-game peers (green band = normal luck, ±2)",
                  legend=dict(orientation="h", y=-0.2))
save(fig, "inspection_candidates_running_luck", height=440)

Luck model check: z across all machines has mean -0.07 and std 1.09 (ideal 0 and 1)

machine  game             zone            designed  actual      z   expected this low by chance  actually this low
M-1158   Desert Gold      Back Forty         10.0%    3.0%  -4.57                          0.00                  1
M-1054   Prairie Thunder  Garage Entrance    11.0%    9.0%  -2.78                          0.85                  2
M-1210   Royal Draw       Main Entrance       5.0%    3.6%  -2.50                          1.90                  3
zoom slider: no


saved luck_z_distribution.html
zoom slider: no


saved inspection_candidates_hold.html
zoom slider: no


saved inspection_candidates_running_luck.html


In [191]:
# Q6: price each candidate action (cost from action_catalog, benefit from our evidence), July 1-30, 2025
PLAN = "B"      # "A" = inspect + signs + promotion   |   "B" = inspect + move 2 machines + promotion
daily   = load("data/1_prepared_views/prepared_daily_view.parquet")
actions = load("data/2_machine_decision/action_catalog.parquet").set_index("action_type_id")
prop    = load("data/1_prepared_views/property_day.parquet")
events  = load("data/4_context/events.parquet")
cal     = load("data/2_machine_decision/calendar_day.parquet")
dq = daily[daily["active"]].copy()
HORIZON = 30

def budget(action_id, qty):
    """Guide formula: qty x (unit cost + daily cost x max(0, 30 - lead days)) / 100."""
    a = actions.loc[action_id]
    return qty * (a["unit_cost_cents"] + a["recurring_cost_cents_per_day"] * max(0, HORIZON - a["lead_days"])) / 100

def benefit_days(action_id):
    return max(0, HORIZON - int(actions.loc[action_id, "lead_days"]))

# ---------- Evidence pulled from the data ----------
# 1) M-1158 payout gap: expected win minus actual win, per day on the floor
m = dq[dq["machine_id"] == "M-1158"]
m1158_gap_per_day = (m["theo_usd"].sum() - m["house_win_usd"].sum()) / len(m)

# 2) Back Forty expected win per day (recent: May-June) and its gap to the floor average
bf = dq[(dq["zone_name"] == "Back Forty") & (dq["gaming_date"].astype(str) >= "2025-05-01")]
bf_theo_per_day = bf.groupby("gaming_date")["theo_usd"].sum().mean()
bf_vs_avg = 1 + zone_norm.loc["Back Forty", "Coin-in / hr"] / 100        # e.g. 0.64 = 36% below average

# 3) Winter Tournament net lift: extra theo + extra F&B margin - extra comp cost, vs normal same-weekday days
prop["date"] = pd.to_datetime(prop["gaming_date"])
event_days = set()
for _, e in events.iterrows():
    event_days |= set(pd.date_range(e["start_date"], e["end_date"]))
cols = ["theo_usd", "comp_cost_usd", "pos_cash_usd", "pos_cogs_usd"]
normal = prop[~prop["date"].isin(event_days)].groupby("local_weekday")[cols].mean()
t = events[events["name"].str.contains("Winter Slot Tournament")].iloc[0]
tdays = prop[(prop["date"] >= pd.to_datetime(t["start_date"])) & (prop["date"] <= pd.to_datetime(t["end_date"]))]
lift = tdays[cols].sum() - normal.loc[tdays["local_weekday"]].sum()
tournament_net = lift["theo_usd"] + (lift["pos_cash_usd"] - lift["pos_cogs_usd"]) - lift["comp_cost_usd"]

# 4) Downtime: Ridgeline lost expected win per day; weak machines' daily shortfall
lost = dt3.assign(tph=dt3["machine_id"].map(per_m["theo"] / per_m["avail"]))
lost["lost"] = lost["down_hrs"] * lost["tph"]
ridgeline_lost_per_day = lost.loc[lost["manufacturer"] == "Ridgeline", "lost"].sum() / lost["gaming_date"].nunique()
floor_lost_per_day = lost["lost"].sum() / lost["gaming_date"].nunique()
worst_weak_per_day = ours.loc[ours["problem"] == "Weak machine", "lost_per_day"].max()

# 5) Moves: 2 empty slots exist in Center Pit (left by earlier moves); median Back Forty machine's theo per day
placements_q6 = load("data/2_machine_decision/placements.parquet")
slots_q6 = load("data/2_machine_decision/floor_locations.parquet")
empty_q6 = slots_q6[(slots_q6["node_type"] == "slot") & ~slots_q6["location_id"].isin(placements_q6.loc[placements_q6["effective_to_utc"].isna(), "location_id"])]
cp_empty = int((empty_q6["zone_id"] == "Z-PIT").sum())
bf_machine_theo = (bf.groupby("machine_id")["theo_usd"].sum() / bf["gaming_date"].nunique()).median()
cp_vs_avg = 1 + zone_norm.loc["Center Pit", "Coin-in / hr"] / 100
move_uplift = cp_vs_avg / bf_vs_avg - 1                                   # Back Forty -> Center Pit (M-1152 move confirmed this size)

print(f"M-1158 payout gap:            ${m1158_gap_per_day:,.0f}/day")
print(f"Back Forty theo (May-Jun):    ${bf_theo_per_day:,.0f}/day   at {bf_vs_avg:.2f} x floor average")
print(f"Winter Tournament net lift:   ${tournament_net:,.0f} over {len(tdays)} days "
      f"(theo +${lift['theo_usd']:,.0f}, F&B margin +${lift['pos_cash_usd'] - lift['pos_cogs_usd']:,.0f}, comps +${lift['comp_cost_usd']:,.0f})")
print(f"Downtime lost per day:        floor ${floor_lost_per_day:,.0f}, Ridgeline ${ridgeline_lost_per_day:,.0f}")
print(f"Worst weak machine shortfall: ${worst_weak_per_day:,.0f}/day")
print(f"Empty Center Pit slots: {cp_empty}   median Back Forty machine theo ${bf_machine_theo:,.0f}/day   Back Forty -> Center Pit uplift {move_uplift:+.0%}\n")

# ---------- Scenarios (assumptions stated: low / central / high) ----------
def close_gap(share):            # Back Forty play lift if signs close `share` of its gap to the floor average
    return (bf_vs_avg + share * (1 - bf_vs_avg)) / bf_vs_avg - 1

d_ins, d_sig = benefit_days("A-INSPECT"), benefit_days("A-WAYFINDING")
d_rep, d_pm  = benefit_days("A-REPLACE-LEASE"), benefit_days("A-PREVENTIVE")
d_mov = benefit_days("A-MOVE-MACHINE")
plan = [
    # name, action id, qty, low, central, high, funded?
    ("Inspect M-1158", "A-INSPECT", 1,
        0,                                              # nothing found
        0.5 * m1158_gap_per_day * d_ins,                # 50% chance a fixable payout fault is found
        m1158_gap_per_day * d_ins, True),               # fault found and fixed on day 2
    ("Signs: Back Forty", "A-WAYFINDING", 1,
        0,                                              # no effect, or players only shift from other zones
        close_gap(0.10) * bf_theo_per_day * d_sig,      # closes 10% of the gap to the floor average
        close_gap(0.20) * bf_theo_per_day * d_sig, PLAN == "A"),   # closes 20%
    ("Move 2 Back Forty machines to empty Center Pit slots", "A-MOVE-MACHINE", cp_empty,
        0.5 * move_uplift * bf_machine_theo * cp_empty * d_mov,   # half the zone gap realised
        1.0 * move_uplift * bf_machine_theo * cp_empty * d_mov,   # full gap (as M-1152 achieved)
        1.2 * move_uplift * bf_machine_theo * cp_empty * d_mov, PLAN == "B"),
    ("Promotion weekend (Jul 18-20)", "A-PROMO", 1,
        0.4 * tournament_net,                           # summer lift 40% of the winter tournament
        0.7 * tournament_net,                           # 70%
        1.0 * tournament_net, True),                    # matches it
    ("Replace 5 weak machines (lease)", "A-REPLACE-LEASE", 5,
        0, 0.5 * worst_weak_per_day * 5 * d_rep, worst_weak_per_day * 5 * d_rep, False),
    ("Preventive maintenance: Ridgeline", "A-PREVENTIVE", 1,
        0.25 * ridgeline_lost_per_day * d_pm, 0.5 * ridgeline_lost_per_day * d_pm,
        0.75 * ridgeline_lost_per_day * d_pm, False),
    ("Extra technician (1 shift)", "A-TECH-SHIFT", 1,
        0.05 * floor_lost_per_day * benefit_days("A-TECH-SHIFT"), 0.15 * floor_lost_per_day * benefit_days("A-TECH-SHIFT"),
        0.30 * floor_lost_per_day * benefit_days("A-TECH-SHIFT"), False),
]
q6 = pd.DataFrame(plan, columns=["action", "id", "qty", "low", "central", "high", "funded"]).set_index("action")
q6["cost"] = [budget(i, q) for i, q in zip(q6["id"], q6["qty"])]
q6["lead_days"] = [int(actions.loc[i, "lead_days"]) for i in q6["id"]]
q6["central_net"] = q6["central"] - q6["cost"]
q6["breakeven_note"] = ""
q6.loc["Signs: Back Forty", "breakeven_note"] = \
    f"break-even if Back Forty play rises {q6.loc['Signs: Back Forty', 'cost'] / (bf_theo_per_day * d_sig):.1%}"

pd.set_option("display.width", 250)
print(q6[["qty", "lead_days", "cost", "low", "central", "high", "central_net", "funded", "breakeven_note"]].round(0).to_string())
spent = q6.loc[q6["funded"], "cost"].sum()
print(f"\nFunded: ${spent:,.0f}   Kept: ${100_000 - spent:,.0f}   Action types used: {q6['funded'].sum()} of 3")

# ---------- Chart: cost vs benefit range
q = q6.sort_values("central")
lab = [("✓ " if f else "") + a for a, f in zip(q.index, q["funded"])]
fig = go.Figure()
for y, (_, row) in zip(lab, q.iterrows()):
    fig.add_scatter(x=[row["low"], row["high"]], y=[y, y], mode="lines", showlegend=False, hoverinfo="skip",
                    line=dict(color=GOLD if row["funded"] else GREY, width=10), opacity=0.45)
fig.add_scatter(x=q["central"], y=lab, mode="markers", name="Benefit: central (bar = low to high)",
                marker=dict(color=[GOLD if f else GREY for f in q["funded"]], size=14, line=dict(color=WHITE, width=1)),
                customdata=np.stack([q["low"], q["high"], q["cost"], q["central_net"]], axis=1).astype(float),
                hovertemplate="%{y}<br>Benefit $%{customdata[0]:,.0f} – $%{x:,.0f} – $%{customdata[1]:,.0f}<br>"
                              "Cost $%{customdata[2]:,.0f} · central net $%{customdata[3]:,.0f}<extra></extra>")
fig.add_scatter(x=q["cost"], y=lab, mode="markers", name="Cost", marker=dict(color=RED, symbol="x", size=13),
                hovertemplate="Cost $%{x:,.0f}<extra></extra>")
fig.update_layout(title=f"Q6: July benefit vs cost (✓ = funded · ${spent:,.0f} spent · ${100_000 - spent:,.0f} kept)",
                  xaxis_title="July 1–30, 2025 (USD)", xaxis_tickprefix="$", legend=dict(orientation="h", y=-0.2))
save(fig, "q6_cost_vs_benefit", height=480)

# How the $100,000 is used
funded = q6[q6["funded"]]
bars = [go.Bar(y=["July budget"], x=[c], name=f"{a}: ${c:,.0f}", orientation="h",
               hovertemplate=f"{a}: ${c:,.0f}<extra></extra>") for a, c in funded["cost"].items()]
bars.append(go.Bar(y=["July budget"], x=[100_000 - spent], name=f"Kept: ${100_000 - spent:,.0f}", orientation="h",
                   marker_color=FAINT, hovertemplate=f"Kept: ${100_000 - spent:,.0f}<extra></extra>"))
fig = go.Figure(bars)
fig.update_layout(barmode="stack", title="How the $100,000 July budget is used", xaxis_tickprefix="$",
                  xaxis_range=[0, 100_000], legend=dict(orientation="h", y=-0.5))
save(fig, "budget_allocation", height=280)

M-1158 payout gap:            $200/day
Back Forty theo (May-Jun):    $13,283/day   at 0.64 x floor average
Winter Tournament net lift:   $86,523 over 3 days (theo +$82,524, F&B margin +$4,667, comps +$668)
Downtime lost per day:        floor $499, Ridgeline $270
Worst weak machine shortfall: $41/day
Empty Center Pit slots: 2   median Back Forty machine theo $284/day   Back Forty -> Center Pit uplift +85%

                                                      qty  lead_days     cost      low  central     high  central_net  funded                            breakeven_note
action                                                                                                                                                                 
Inspect M-1158                                          1          1   1200.0      0.0   2897.0   5794.0       1697.0    True                                          
Signs: Back Forty                                       1         10   7500.0      0.0 

saved q6_cost_vs_benefit.html
zoom slider: no


saved budget_allocation.html


### How sure are we? Sensitivity of the plan

Every benefit above depends on an assumption: the chance M-1158 has a real fault, how much of the winter tournament lift repeats in summer, and how much of the Center Pit advantage moved machines pick up. The next chart changes one assumption at a time and shows how July net benefit (benefit minus cost) changes.

- **Spider (top):** each line moves one assumption from 0% to 200% of its base value. A steeper line means that assumption matters more. Where a line crosses $0, the plan stops paying for itself.
- **Tornado (bottom):** each bar sets one assumption to its low or high case. The longest bar is the biggest risk.
- **Break-even table:** the value each assumption needs for its action to pay for itself, compared with our base value.

In [192]:
# Q6 sensitivity, clean version: Plan B only (tornado) + how much room each action has before it stops paying
import plotly.graph_objects as go
from plotly.subplots import make_subplots

p = "moves"
b0 = plan_net(base, p)
rows = sorted(((k, plan_net({**base, k: low_high[k][0]}, p), plan_net({**base, k: low_high[k][1]}, p)) for k in relevant[p]),
              key=lambda r: abs(r[2] - r[1]))
labels = {"P(fixable fault at M-1158)": "M-1158 has a fixable fault (0% – 100%)",
          "Promo: summer lift vs winter": "Summer promo lift vs winter (40% – 100%)",
          "Moves: realised share of zone gap": "Moved machines reach Center Pit level (50% – 120%)",
          "Back Forty theo per day": "Back Forty play level (−20% – +20%)"}

fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.22,
                    subplot_titles=["If one assumption is wrong, what is July net benefit?",
                                    "Safety margin: how much weaker before each action loses money"])
for k, lo, hi in rows:
    fig.add_trace(go.Scatter(x=[lo, hi], y=[labels[k]] * 2, mode="lines", line=dict(color="#c9ced6", width=14),
                             showlegend=False, hoverinfo="skip"), 1, 1)
    fig.add_trace(go.Scatter(x=[lo], y=[labels[k]], mode="markers+text", marker=dict(color="#d1495b", size=14),
                             text=[f"${lo/1000:,.0f}K"], textposition="middle left", showlegend=False,
                             hovertemplate="low case: $%{x:,.0f}<extra></extra>"), 1, 1)
    fig.add_trace(go.Scatter(x=[hi], y=[labels[k]], mode="markers+text", marker=dict(color="#2e86ab", size=14),
                             text=[f"${hi/1000:,.0f}K"], textposition="middle right", showlegend=False,
                             hovertemplate="high case: $%{x:,.0f}<extra></extra>"), 1, 1)
fig.add_vline(x=b0, line_dash="dash", line_color="black", row=1, col=1,
              annotation_text=f"base ${b0/1000:,.0f}K", annotation_position="top")
fig.add_vline(x=0, line_color="#d1495b", line_width=1, row=1, col=1)
fig.update_xaxes(title_text="July net benefit ($)", tickprefix="$", row=1, col=1)

bb = be[be["action"] != "Signs: Back Forty"].sort_values("safety margin (base ÷ break-even)")
fig.add_trace(go.Bar(x=bb["safety margin (base ÷ break-even)"], y=bb["action"], orientation="h", marker_color="#3a7d44",
                     text=[f"{v:.1f}×" for v in bb["safety margin (base ÷ break-even)"]], textposition="outside",
                     showlegend=False, hovertemplate="%{y}: %{x:.1f}× margin<extra></extra>"), 1, 2)
fig.add_vline(x=1, line_color="#d1495b", row=1, col=2, annotation_text="break-even", annotation_position="bottom")
fig.update_xaxes(title_text="base assumption ÷ break-even value", range=[0, bb["safety margin (base ÷ break-even)"].max() * 1.3], row=1, col=2)

fig.update_layout(height=420, width=1300, template="simple_white", font=dict(size=13),
                  title="Q6 Plan B (inspect M-1158 + move 2 machines + Jul 18–20 promotion): how sure are we?",
                  margin=dict(l=20, r=40, t=90, b=60))
fig.show()

NameError: name 'plan_net' is not defined